# TourFold v4 · Çok çekirdekli LKH karşılaştırması

Pratikte LKH çoğu zaman çok çekirdekte, her çekirdekte farklı tohumla koşturulur ve en iyi tur alınır. Bu defter v2–v3'teki 51 örneğin her birini 6 paralel LKH koşusuyla iki yoldan çözer:

- **Taban:** 6 LKH-3 POPMUSIC koşusu baştan sona.
- **TourFold:** aynı 6 koşu ilk denemeden sonra durur. TourFold-v2 bu 6 turu bir topluluk gibi okur; her koşu kendi turundan, TourFold'un ilk 3 adayı + kendi POPMUSIC listesiyle devam eder.

İlk deneme iki tarafta aynıdır; tek fark aday listesidir. Bu düzende topluluk bedava gelir (v3'te tek çekirdekte bütçenin %15–46'sını yiyordu).

- **Çalıştırma:** *Çalışma zamanı ▸ Tümünü çalıştır*, ardından Drive izni. 12 vCPU'lu bir çalışma zamanı seçin; GPU türü önemsiz (GPU yalnız modelin tahmini için kullanılır).
- **Süre:** yaklaşık 4 saat (12 vCPU). Sekmeyi açık bırakın; aynı anda başka bir TourFold defteri çalıştırmayın.
- **Sonuçlar:** `MyDrive/TourFold_TSP/v4` — rapor `RAPOR_v4.md`, ilerleme ve kalan süre `log_v4.txt` / `status_v4.json`. Bitince `TourFold_v4_sonuclar.zip` Drive'a kaydedilir ve tarayıcınıza iner.
- **Bir adım hata verirse** sonraki deney adımları atlanır, ama rapor ve sonuç paketi yine yazılır.
- **Bağlantı koparsa:** *Tümünü çalıştır*'ı yeniden seçin; biten örnekler atlanır.
- **Gerekenler:** önceki çalıştırmaların Drive'daki verisi (değerlendirme örnekleri ve v2'nin TourFold-v2 modeli). Yeni veri ya da eğitim yok.

## 1 · Yardımcı modüller
LKH'yi çalıştıran (`tsplab`) ve TourFold modelini tanımlayan (`tsp_model`) iki modülü diske yazar; paralel LKH süreçleri onları buradan içe aktarır. Okumanız gerekmez. Birkaç saniye.

In [ ]:
#@title Yardımcı modüller (kodu görmek için hücreye çift tıklayın)
# Bu hücreyi okumanız gerekmez: iki yardımcı modülü diske yazar (paralel LKH süreçleri onları içe aktarır).
import os, sys, importlib

TEST_MODE = os.environ.get("TOURFOLD_TEST") == "1"          # yalnızca yerel testler için
LOCAL = os.environ["TOURFOLD_LOCAL"] if TEST_MODE else "/content/tourfold_local"
os.makedirs(LOCAL, exist_ok=True)
MODULES = {"tsplab.py": r'''"""
tsplab - utilities for "learned guidance + LKH-3" TSP experiments.

  * instance generation (uniform / clustered / mixed), integer scaling
  * TSPLIB I/O: problem, tour, parameter, LKH candidate and penalty (pi) files
  * LKH-3 download/build, runner, output parser (anytime trace)
  * Held-Karp 1-tree bound with LKH's 1-tree definition (unit checks)
  * candidate-set coverage metrics
  * dataset shards for variable-size instances
  * process-parallel map with error capture
"""
from __future__ import annotations

import glob
import gzip
import io
import json
import math
import os
import re
import shutil
import subprocess
import tarfile
import tempfile
import time
import traceback
import urllib.request

import numpy as np

try:
    from tqdm.auto import tqdm
except Exception:  # pragma: no cover
    def tqdm(x, **kwargs):
        return x

COORD_SCALE = 1_000_000          # synthetic instances: unit square -> integer grid
LKH_PRECISION = 100              # LKH default PRECISION
CAND_K = 5                       # LKH default MAX_CANDIDATES

# Known optimal tour lengths (TSPLIB, symmetric, 1k-20k nodes).
TSPLIB_OPT = {
    "pr1002": 259045, "si1032": 92650, "u1060": 224094, "vm1084": 239297,
    "pcb1173": 56892, "d1291": 50801, "rl1304": 252948, "rl1323": 270199,
    "nrw1379": 56638, "fl1400": 20127, "u1432": 152970, "fl1577": 22249,
    "d1655": 62128, "vm1748": 336556, "u1817": 57201, "rl1889": 316536,
    "d2103": 80450, "u2152": 64253, "u2319": 234256, "pr2392": 378032,
    "pcb3038": 137694, "fl3795": 28772, "fnl4461": 182566, "rl5915": 565530,
    "rl5934": 556045, "pla7397": 23260728, "rl11849": 923288,
    "usa13509": 19982859, "brd14051": 469385, "d15112": 1573084,
    "d18512": 645238, "dsj1000": 18660188,
}


# ----------------------------------------------------------------------------- env
def env_info():
    info = {"nproc": os.cpu_count()}
    try:
        with open("/proc/meminfo") as f:
            kb = int(re.search(r"MemTotal:\s+(\d+)", f.read()).group(1))
        info["ram_gb"] = round(kb / 1024 ** 2, 1)
    except Exception:
        info["ram_gb"] = None
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=20).stdout.strip()
        info["gpu"] = out or None
    except Exception:
        info["gpu"] = None
    info["numpy"] = np.__version__
    return info


# ---------------------------------------------------------------------- generation
def normalize_unit(xy):
    xy = np.asarray(xy, dtype=np.float64)
    mn = xy.min(0)
    span = float((xy.max(0) - mn).max())
    return (xy - mn) / (span if span > 0 else 1.0)


def gen_uniform(n, rng):
    return rng.random((n, 2))


def gen_clustered(n, rng, n_clusters=None, sigma=None):
    c = int(n_clusters if n_clusters is not None else rng.integers(3, 16))
    s = float(sigma if sigma is not None else math.exp(rng.uniform(math.log(0.02), math.log(0.08))))
    centers = rng.random((c, 2))
    lab = rng.integers(0, c, n)
    return normalize_unit(centers[lab] + rng.normal(0.0, s, (n, 2)))


def gen_instance(n, dist, rng):
    if dist == "uniform":
        return gen_uniform(n, rng)
    if dist == "clustered":
        return gen_clustered(n, rng)
    if dist == "mixed":
        return gen_uniform(n, rng) if rng.random() < 0.5 else gen_clustered(n, rng)
    raise ValueError(dist)


def to_int_coords(xy01, scale=COORD_SCALE):
    return np.rint(np.asarray(xy01, dtype=np.float64) * scale).astype(np.int64)


# ------------------------------------------------------------------------ distances
def edge_len(a, b, ew="EUC_2D"):
    d = np.sqrt(((np.asarray(a, dtype=np.float64) - np.asarray(b, dtype=np.float64)) ** 2).sum(-1))
    if ew == "EUC_2D":
        return np.floor(d + 0.5)
    if ew == "CEIL_2D":
        return np.ceil(d)
    if ew in ("FLOAT", "EXACT"):
        return d
    raise ValueError(f"unsupported EDGE_WEIGHT_TYPE {ew}")


def tour_length(xy, tour, ew="EUC_2D"):
    t = np.asarray(tour)
    xy = np.asarray(xy)
    return float(edge_len(xy[t], xy[np.roll(t, -1)], ew).sum())


def is_valid_tour(tour, n):
    if tour is None:
        return False
    t = np.asarray(tour)
    return t.shape == (n,) and np.array_equal(np.sort(t), np.arange(n))


def gap_pct(cost, ref):
    return 100.0 * (cost - ref) / ref


# --------------------------------------------------------------------- TSPLIB I/O
def _open_text(path):
    if str(path).endswith(".gz"):
        return io.TextIOWrapper(gzip.open(path), encoding="latin-1")
    return open(path, encoding="latin-1")


def _is_int_token(s):
    try:
        int(s)
        return True
    except ValueError:
        return False


def write_tsp(path, xy, name="instance", ew="EUC_2D", comment=None):
    xy = np.asarray(xy)
    head = [f"NAME : {name}", "TYPE : TSP"]
    if comment:
        head.append(f"COMMENT : {comment}")
    head += [f"DIMENSION : {len(xy)}", f"EDGE_WEIGHT_TYPE : {ew}", "NODE_COORD_SECTION"]
    if np.issubdtype(xy.dtype, np.integer):
        body = "\n".join(f"{i + 1} {x} {y}" for i, (x, y) in enumerate(xy.tolist()))
    else:
        body = "\n".join(f"{i + 1} {x!r} {y!r}" for i, (x, y) in enumerate(xy.tolist()))
    with open(path, "w") as f:
        f.write("\n".join(head) + "\n" + body + "\nEOF\n")


def read_tsplib(path):
    hdr, ids, coords = {}, [], []
    section = None
    with _open_text(path) as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.upper() == "EOF":
                break
            up = line.upper()
            if section == "NODE_COORD_SECTION":
                parts = line.split()
                if len(parts) >= 3 and _is_int_token(parts[0]):
                    ids.append(int(parts[0]))
                    coords.append((float(parts[1]), float(parts[2])))
                    continue
                section = None
            if "_SECTION" in up.split(":")[0]:
                section = up.split(":")[0].strip()
                continue
            if section is not None:
                continue
            if ":" in line:
                k, v = line.split(":", 1)
                hdr[k.strip().upper()] = v.strip()
    xy = np.array(coords, dtype=np.float64)
    if ids:
        xy = xy[np.argsort(np.array(ids))]
    return {
        "name": hdr.get("NAME", os.path.basename(str(path)).split(".")[0]),
        "type": hdr.get("TYPE"),
        "dimension": int(hdr.get("DIMENSION", len(xy))),
        "edge_weight_type": hdr.get("EDGE_WEIGHT_TYPE"),
        "comment": hdr.get("COMMENT"),
        "xy": xy,
    }


def write_par(path, params):
    with open(path, "w") as f:
        for k, v in params.items():
            if v is None:
                continue
            for x in (v if isinstance(v, (list, tuple)) else [v]):
                if isinstance(x, bool):
                    x = "YES" if x else "NO"
                f.write(f"{k} = {x}\n")


def read_tour(path):
    vals, in_sec = [], False
    with _open_text(path) as f:
        for line in f:
            s = line.strip()
            if not in_sec:
                if s.upper().startswith("TOUR_SECTION"):
                    in_sec = True
                continue
            for tok in s.split():
                if tok.upper() == "EOF":
                    break
                v = int(tok)
                if v == -1:
                    return np.array(vals, dtype=np.int64) - 1
                vals.append(v)
    return np.array(vals, dtype=np.int64) - 1


def write_tour(path, tour, name="tour", length=None):
    t = np.asarray(tour, dtype=np.int64) + 1
    head = [f"NAME : {name}"]
    if length is not None:
        head.append(f"COMMENT : Length = {int(length)}")
    head += ["TYPE : TOUR", f"DIMENSION : {len(t)}", "TOUR_SECTION"]
    with open(path, "w") as f:
        f.write("\n".join(head) + "\n" + "\n".join(map(str, t.tolist())) + "\n-1\nEOF\n")


def write_candidates(path, cand, alpha=None, dad=None):
    """LKH CANDIDATE_FILE. cand: (n,K) 0-based node ids, -1 padded, best first.
    alpha: same shape integer ranks/scores (default rank*100). dad: (n,) 0-based or -1."""
    cand = np.asarray(cand, dtype=np.int64)
    n = len(cand)
    lines = [str(n)]
    for i in range(n):
        row = cand[i]
        valid = row >= 0
        row = row[valid]
        if alpha is None:
            al = np.arange(len(row), dtype=np.int64) * 100
        else:
            al = np.asarray(alpha[i], dtype=np.int64)[valid]
        d = 0 if dad is None or dad[i] < 0 else int(dad[i]) + 1
        pairs = np.empty(2 * len(row), dtype=np.int64)
        pairs[0::2] = row + 1
        pairs[1::2] = al
        lines.append(f"{i + 1} {d} {len(row)} " + " ".join(map(str, pairs.tolist())))
    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n-1\nEOF\n")


def read_candidates(path, k_pad=None):
    with open(path) as f:
        toks = f.read().split()
    n = int(toks[0])
    pos = 1
    lists, alists = [None] * n, [None] * n
    dad = np.full(n, -1, dtype=np.int64)
    while pos < len(toks) and _is_int_token(toks[pos]):
        tid = int(toks[pos])
        pos += 1
        if tid == -1:
            break
        d, cnt = int(toks[pos]), int(toks[pos + 1])
        pos += 2
        v = np.array(toks[pos:pos + 2 * cnt], dtype=np.int64).reshape(cnt, 2)
        pos += 2 * cnt
        order = np.argsort(v[:, 1], kind="stable")
        lists[tid - 1], alists[tid - 1] = v[order, 0] - 1, v[order, 1]
        dad[tid - 1] = d - 1
    K = max([len(x) for x in lists if x is not None] + [0])
    K = max(K, k_pad or 0)
    C = np.full((n, K), -1, dtype=np.int64)
    A = np.full((n, K), -1, dtype=np.int64)
    for i, x in enumerate(lists):
        if x is not None and len(x):
            C[i, :len(x)] = x
            A[i, :len(x)] = alists[i]
    return {"n": n, "cand": C, "alpha": A, "dad": dad}


def write_pi(path, pi):
    pi = np.rint(np.asarray(pi, dtype=np.float64)).astype(np.int64)
    with open(path, "w") as f:
        f.write(f"{len(pi)}\n" + "\n".join(f"{i + 1} {v}" for i, v in enumerate(pi.tolist())) + "\n-1\nEOF\n")


def read_pi(path):
    with open(path) as f:
        toks = f.read().split()
    n = int(toks[0])
    pi = np.zeros(n, dtype=np.int64)
    pos = 1
    for _ in range(n):
        i, v = int(toks[pos]), int(toks[pos + 1])
        pos += 2
        pi[i - 1] = v
    return pi


# -------------------------------------------------------------------------- LKH
LKH_BASES = ["http://webhotel4.ruc.dk/~keld/research/LKH-3/",
             "https://webhotel4.ruc.dk/~keld/research/LKH-3/"]
LKH_FALLBACK_VERSIONS = ["3.0.13", "3.0.12", "3.0.11", "3.0.10", "3.0.9", "3.0.8"]


def _urlget(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (tsplab)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()


def _extract(tgz, dest):
    with tarfile.open(tgz) as tf:
        try:
            tf.extractall(dest, filter="data")
        except TypeError:
            tf.extractall(dest)


def build_lkh(dest="/content/lkh", version=None, verbose=True):
    """Download the latest LKH-3 from the author's site and compile it. Returns the binary path."""
    os.makedirs(dest, exist_ok=True)
    found = sorted(glob.glob(os.path.join(dest, "LKH-3*", "LKH")))
    if found and version is None:
        return found[-1]
    versions = [version] if version else []
    if not versions:
        for base in LKH_BASES:
            try:
                html = _urlget(base, 30).decode("latin-1")
                vs = sorted({tuple(map(int, m)) for m in re.findall(r"LKH-3\.(\d+)\.(\d+)\.tgz", html)},
                            reverse=True)
                versions = [f"3.{a}.{b}" for a, b in vs]
                if versions:
                    break
            except Exception as e:  # noqa: BLE001
                if verbose:
                    print("index fetch failed:", base, e)
        versions += [v for v in LKH_FALLBACK_VERSIONS if v not in versions]
    last = None
    for v in versions:
        for base in LKH_BASES:
            try:
                data = _urlget(f"{base}LKH-{v}.tgz", 180)
            except Exception as e:  # noqa: BLE001
                last = e
                continue
            tgz = os.path.join(dest, f"LKH-{v}.tgz")
            with open(tgz, "wb") as f:
                f.write(data)
            _extract(tgz, dest)
            src = os.path.join(dest, f"LKH-{v}")
            r = subprocess.run(["make"], cwd=src, capture_output=True, text=True)
            binp = os.path.join(src, "LKH")
            if r.returncode != 0 or not os.path.exists(binp):
                raise RuntimeError(f"LKH build failed:\n{r.stdout[-3000:]}\n{r.stderr[-3000:]}")
            if verbose:
                print(f"LKH-{v} built at {binp}")
            return binp
    raise RuntimeError(f"Could not download LKH-3 ({last})")


_RX = {
    "lb": re.compile(r"Lower bound\s*=\s*(-?[\d.]+)"),
    "ascent": re.compile(r"Ascent time\s*=\s*([\d.]+)\s*sec"),
    "pre": re.compile(r"Preprocessing time\s*=\s*([\d.]+)\s*sec"),
    "imp": re.compile(r"^\*\s*(\d+):\s*Cost\s*=\s*(-?\d+)[^\n]*?Time\s*=\s*([\d.]+)\s*sec", re.M),
    "run": re.compile(r"^Run\s+(\d+):\s*Cost\s*=\s*(-?\d+)[^\n]*?Time\s*=\s*([\d.]+)\s*sec", re.M),
    "costmin": re.compile(r"Cost\.min\s*=\s*(-?\d+)"),
    "timetotal": re.compile(r"Time\.total\s*=\s*([\d.]+)\s*sec"),
    "cand": re.compile(r"Cand\.min\s*=\s*(\d+),\s*Cand\.avg\s*=\s*([\d.]+),\s*Cand\.max\s*=\s*(\d+)"),
    "trials": re.compile(r"Trials\.min\s*=\s*(\d+)"),
}


def parse_lkh_output(text):
    def f1(key, cast=float):
        m = _RX[key].search(text)
        return cast(m.group(1)) if m else None

    out = {
        "lower_bound": f1("lb"),
        "ascent_time": f1("ascent"),
        "pre_time": f1("pre"),
        "cost_min": f1("costmin", int),
        "time_total": f1("timetotal"),
        "trials": f1("trials", int),
        "events": [(int(a), int(b), float(c)) for a, b, c in _RX["imp"].findall(text)],
        "runs": [(int(a), int(b), float(c)) for a, b, c in _RX["run"].findall(text)],
    }
    m = _RX["cand"].search(text)
    out["cand_stats"] = (int(m.group(1)), float(m.group(2)), int(m.group(3))) if m else None
    return out


def run_lkh(lkh_bin, problem_file, params, workdir, timeout=None, keep_stdout=False):
    os.makedirs(workdir, exist_ok=True)
    par_path = os.path.join(workdir, "run.par")
    p = {"PROBLEM_FILE": problem_file, "RUNS": 1, "TRACE_LEVEL": 1}
    p.update(params or {})
    p.setdefault("OUTPUT_TOUR_FILE", os.path.join(workdir, "run.tour"))
    write_par(par_path, p)
    t0, t_start = time.perf_counter(), time.time()      # t_start: wall clock, comparable across processes
    try:
        proc = subprocess.run([lkh_bin, par_path], cwd=workdir, stdin=subprocess.DEVNULL,
                              capture_output=True, text=True, errors="replace", timeout=timeout)
        rc, so, se, to = proc.returncode, proc.stdout, proc.stderr, False
    except subprocess.TimeoutExpired as e:
        so = e.stdout.decode("latin-1", "replace") if isinstance(e.stdout, bytes) else (e.stdout or "")
        se = e.stderr.decode("latin-1", "replace") if isinstance(e.stderr, bytes) else (e.stderr or "")
        rc, to = None, True
    wall = time.perf_counter() - t0
    res = parse_lkh_output(so)
    res.update(wall=wall, returncode=rc, timed_out=to, stderr=(se or "")[-2000:], t_start=t_start)
    res["stdout"] = so if keep_stdout else so[-4000:]
    tp = p["OUTPUT_TOUR_FILE"]
    res["tour"] = read_tour(tp) if os.path.exists(tp) else None
    return res


def lkh_solve(lkh_bin, xy=None, problem_file=None, ew="EUC_2D", params=None, workroot=None,
              want_cands=False, want_pi=False, cand_file=None, pi_file=None, timeout=None,
              keep_stdout=False):
    """Run LKH on coordinates (written to a temp TSPLIB file) or on an existing problem file.
    cand_file / pi_file: existing files to be READ by LKH (learned guidance).
    want_cands / want_pi: let LKH WRITE its own candidate set / penalties and return them."""
    os.makedirs(workroot or tempfile.gettempdir(), exist_ok=True)
    wd = tempfile.mkdtemp(dir=workroot, prefix="lkh_")
    try:
        if problem_file is None:
            problem_file = os.path.join(wd, "p.tsp")
            write_tsp(problem_file, xy, ew=ew)
        p = dict(params or {})
        out_cand = out_pi = None
        if cand_file is not None:
            p["CANDIDATE_FILE"] = cand_file
        elif want_cands:
            out_cand = p["CANDIDATE_FILE"] = os.path.join(wd, "out.cand")
        if pi_file is not None:
            p["PI_FILE"] = pi_file
        elif want_pi:
            out_pi = p["PI_FILE"] = os.path.join(wd, "out.pi")
        res = run_lkh(lkh_bin, problem_file, p, wd, timeout=timeout, keep_stdout=keep_stdout)
        if out_cand and os.path.exists(out_cand):
            res["cand"] = read_candidates(out_cand, k_pad=p.get("MAX_CANDIDATES", CAND_K))
        if out_pi and os.path.exists(out_pi):
            res["pi"] = read_pi(out_pi)
        return res
    finally:
        shutil.rmtree(wd, ignore_errors=True)


def anytime_curve(res, ref, budgets):
    """Gap (%) of the best tour found within each total-time budget (preprocessing + search)."""
    pre = res.get("pre_time") or 0.0
    ev = sorted(res.get("events") or [], key=lambda e: e[2])
    out = []
    for b in budgets:
        best = None
        for _, c, t in ev:
            if pre + t <= b:
                best = c if best is None else min(best, c)
        out.append(gap_pct(best, ref) if best is not None else np.nan)
    return out


TSPLIB_BASES = ["https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/",
                "http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/"]


def _valid_tsp(path):
    try:
        if os.path.getsize(path) == 0:
            return False
        inst = read_tsplib(path)
        return inst["dimension"] > 0 and (len(inst["xy"]) == inst["dimension"] or inst["edge_weight_type"] == "EXPLICIT")
    except Exception:  # noqa: BLE001
        return False


def _local_tsplib(dest, names):
    """{name: dest/name.tsp} for every name found under dest as a valid .tsp or .tsp.gz (any subfolder).
    Empty files (left behind by interrupted downloads) are removed so that they never count as present."""
    have = {}
    for nm in names:
        out = os.path.join(dest, nm + ".tsp")
        if os.path.exists(out):
            if _valid_tsp(out):
                have[nm] = out
                continue
            os.remove(out)
        for cand in sorted(glob.glob(os.path.join(dest, "**", nm + ".tsp*"), recursive=True)):
            if cand == out or not (cand.endswith(".tsp") or cand.endswith(".tsp.gz")):
                continue
            try:
                if os.path.getsize(cand) == 0:
                    os.remove(cand)
                    continue
                if cand.endswith(".gz"):
                    with gzip.open(cand, "rb") as fi, open(out + ".part", "wb") as fo:
                        shutil.copyfileobj(fi, fo)
                    os.replace(out + ".part", out)
                else:
                    shutil.copy(cand, out)
                if _valid_tsp(out):
                    have[nm] = out
                    break
                os.remove(out)
            except Exception:  # noqa: BLE001
                continue
    return have


def download_tsplib(dest, names=None, verbose=True, timeout=30, bases=None):
    """TSPLIB problems as {name: path.tsp}.
    1) files already under dest (.tsp, .tsp.gz or ALL_tsp.tar.gz, e.g. copied there by the user);
    2) the official TSPLIB95 server. Each address is first probed with one small file and a short timeout,
       so an unreachable server costs about a minute instead of one long timeout per file."""
    os.makedirs(dest, exist_ok=True)
    names = list(names or TSPLIB_OPT)
    tgz = os.path.join(dest, "ALL_tsp.tar.gz")
    if os.path.exists(tgz):
        if os.path.getsize(tgz) == 0:
            os.remove(tgz)                      # left behind by an interrupted download
        else:
            try:
                _extract(tgz, dest)
            except Exception as e:  # noqa: BLE001
                if verbose:
                    print("could not extract", tgz, e)
    have = _local_tsplib(dest, names)
    if len(have) == len(names):
        return have
    for base in (bases or TSPLIB_BASES):
        probe = next(nm for nm in names if nm not in have)
        try:
            data = _urlget(base + probe + ".tsp.gz", timeout)
        except Exception as e:  # noqa: BLE001
            if verbose:
                print("TSPLIB server not answering at", base, "-", e)
            continue
        with open(os.path.join(dest, probe + ".tsp.gz"), "wb") as f:
            f.write(data)
        try:
            data = _urlget(base + "ALL_tsp.tar.gz", max(timeout, 120))
            with open(tgz + ".part", "wb") as f:
                f.write(data)
            os.replace(tgz + ".part", tgz)
            _extract(tgz, dest)
        except Exception as e:  # noqa: BLE001
            if verbose:
                print("ALL_tsp.tar.gz not available, fetching single files:", e)
        have = _local_tsplib(dest, names)
        for nm in [m for m in names if m not in have]:
            try:
                data = _urlget(base + nm + ".tsp.gz", timeout)
                with open(os.path.join(dest, nm + ".tsp.gz"), "wb") as f:
                    f.write(data)
            except Exception as e:  # noqa: BLE001
                if verbose:
                    print("could not get", nm, e)
        break
    return _local_tsplib(dest, names)


def time_to_gap(res, ref, target_pct):
    """Earliest total time (preprocessing + search) at which the gap reaches target_pct; inf if never."""
    pre = res.get("pre_time") or 0.0
    for _, c, t in sorted(res.get("events") or [], key=lambda e: e[2]):
        if gap_pct(c, ref) <= target_pct + 1e-12:
            return pre + t
    return float("inf")


# ------------------------------------------------------------- Held-Karp 1-tree
def _row_cost(xy, u, ew, precision, pi):
    r = edge_len(xy[u], xy, ew) * precision
    if pi is not None:
        r = r + pi[u] + pi
    return r


def held_karp_bound(xy, pi=None, ew="EUC_2D", precision=1.0, chunk=256):
    """w(pi) with LKH's minimum 1-tree: MST + the longest second-nearest edge among MST leaves.
    Transformed cost C'_ij = precision*d_ij + pi_i + pi_j.  Returns bound in distance units."""
    xy = np.asarray(xy, dtype=np.float64)
    n = len(xy)
    pia = None if pi is None else np.asarray(pi, dtype=np.float64)
    in_tree = np.zeros(n, dtype=bool)
    best = np.full(n, np.inf)
    parent = np.full(n, -1, dtype=np.int64)
    best[0] = 0.0
    total = 0.0
    for _ in range(n):
        u = int(np.argmin(best))
        total += best[u]
        in_tree[u] = True
        best[u] = np.inf
        row = _row_cost(xy, u, ew, precision, pia)
        upd = (~in_tree) & (row < best)
        best[upd] = row[upd]
        parent[upd] = u
    deg = np.bincount(parent[parent >= 0], minlength=n) + (parent >= 0)
    nbr = parent.copy()
    root_child = np.where(parent == 0)[0]
    if deg[0] == 1:
        nbr[0] = root_child[0]
    leaves = np.where(deg == 1)[0]
    best_val, best_leaf, best_to = -np.inf, -1, -1
    for s in range(0, len(leaves), chunk):
        L = leaves[s:s + chunk]
        D = edge_len(xy[L][:, None, :], xy[None, :, :], ew) * precision
        if pia is not None:
            D = D + pia[L][:, None] + pia[None, :]
        D[np.arange(len(L)), L] = np.inf
        D[np.arange(len(L)), nbr[L]] = np.inf
        j = D.argmin(1)
        v = D[np.arange(len(L)), j]
        k = int(v.argmax())
        if v[k] > best_val:
            best_val, best_leaf, best_to = float(v[k]), int(L[k]), int(j[k])
    deg = deg.copy()
    deg[best_leaf] += 1
    deg[best_to] += 1
    w = total + best_val - (2.0 * pia.sum() if pia is not None else 0.0)
    return {"bound": w / precision, "degree": deg, "parent": parent, "special": (best_leaf, best_to)}


# ----------------------------------------------------------------- candidates
def knn(xy, k):
    from scipy.spatial import cKDTree
    xy = np.asarray(xy, dtype=np.float64)
    n = len(xy)
    k = min(k, n - 1)
    d, idx = cKDTree(xy).query(xy, k=k + 1)
    self_mask = idx == np.arange(n)[:, None]
    no_self = ~self_mask.any(1)
    self_mask[no_self, -1] = True
    keep = ~self_mask
    return idx[keep].reshape(n, k), d[keep].reshape(n, k)


def tour_succ_pred(tour):
    t = np.asarray(tour)
    succ = np.empty(len(t), dtype=np.int64)
    pred = np.empty(len(t), dtype=np.int64)
    succ[t] = np.roll(t, -1)
    pred[t] = np.roll(t, 1)
    return succ, pred


def tour_edge_labels(nbr_idx, tour):
    """(n,k) bool: is edge (i, nbr_idx[i,j]) in the tour (either direction)."""
    succ, pred = tour_succ_pred(tour)
    return (nbr_idx == succ[:, None]) | (nbr_idx == pred[:, None])


def candidate_coverage(cand, tour, ks=None):
    cand = np.asarray(cand)
    n, K = cand.shape
    succ, pred = tour_succ_pred(tour)
    out = {}
    for k in (ks or range(1, K + 1)):
        c = cand[:, :k]
        in_s = (c == succ[:, None]).any(1)
        in_p = (c == pred[:, None]).any(1)
        und = in_s | in_p[succ]
        out[int(k)] = {"directed": float((in_s.sum() + in_p.sum()) / (2 * n)),
                       "undirected": float(und.mean()),
                       "missing": int(n - und.sum())}
    return out


def pad_cols(a, K, fill=-1):
    a = np.asarray(a)
    if a.shape[1] >= K:
        return a[:, :K]
    return np.concatenate([a, np.full((a.shape[0], K - a.shape[1]), fill, dtype=a.dtype)], 1)


# --------------------------------------------------------------------- shards
def save_shard(path, records):
    ns = np.array([int(r["n"]) for r in records], dtype=np.int64)
    data = {"offsets": np.concatenate([[0], np.cumsum(ns)])}
    for k, v0 in records[0].items():
        if k == "n":
            continue
        if isinstance(v0, np.ndarray) and v0.ndim >= 1 and len(v0) == records[0]["n"]:
            data["node__" + k] = np.concatenate([np.asarray(r[k]) for r in records], 0)
        else:
            vals = [np.nan if r.get(k) is None else r.get(k) for r in records]
            data["scal__" + k] = np.array(vals)
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    tmp = path[:-4] + ".tmp.npz" if path.endswith(".npz") else path + ".tmp.npz"
    np.savez_compressed(tmp, **data)
    os.replace(tmp, path)


class Shard:
    def __init__(self, path):
        z = np.load(path, allow_pickle=False)
        self.off = z["offsets"]
        self.node = {k[6:]: z[k] for k in z.files if k.startswith("node__")}
        self.scal = {k[6:]: z[k] for k in z.files if k.startswith("scal__")}

    def __len__(self):
        return len(self.off) - 1

    def __getitem__(self, i):
        a, b = int(self.off[i]), int(self.off[i + 1])
        r = {k: v[a:b] for k, v in self.node.items()}
        r.update({k: (v[i].item() if getattr(v[i], "ndim", 1) == 0 else v[i]) for k, v in self.scal.items()})
        r["n"] = b - a
        return r


def load_shards(pattern):
    out = []
    for p in sorted(glob.glob(pattern)):
        s = Shard(p)
        out.extend(s[i] for i in range(len(s)))
    return out


# ---------------------------------------------------------------- parallel map
def _safe_call(args):
    fn, kw = args
    try:
        return fn(**kw)
    except Exception as e:  # noqa: BLE001
        return {"error": repr(e), "traceback": traceback.format_exc()}


def pmap(fn, kwargs_list, workers=None, desc=None, ctx="fork"):
    """Process-parallel map; exceptions are returned as {'error': ...} instead of raised."""
    import concurrent.futures as cf
    import multiprocessing as mp
    workers = max(1, workers or os.cpu_count() or 1)
    if workers == 1 or len(kwargs_list) <= 1:
        return [_safe_call((fn, kw)) for kw in tqdm(kwargs_list, desc=desc)]
    out = [None] * len(kwargs_list)
    with cf.ProcessPoolExecutor(max_workers=workers, mp_context=mp.get_context(ctx)) as ex:
        futs = {ex.submit(_safe_call, (fn, kw)): i for i, kw in enumerate(kwargs_list)}
        for f in tqdm(cf.as_completed(futs), total=len(futs), desc=desc):
            out[futs[f]] = f.result()
    return out


def errors_of(results):
    return [r for r in results if isinstance(r, dict) and "error" in r]


def call_task(fn, kw):
    """Run fn(**kw). Lets a single pool run different task functions; picklable under 'spawn'
    (functions defined inside a notebook cell are not)."""
    return fn(**kw)


# ------------------------------------------------------------ task functions
def task_record(lkh_bin, workroot, seed, xy=None, problem_file=None, ew="EUC_2D", params=None,
                dist=None, name=None, require_files=True, ensemble_M=0):
    """Solve one instance with LKH and keep: tour, alpha-candidates, penalties (pi), timings.
    ensemble_M > 0 also stores a TourFold ensemble: M single-trial tours on shared POPMUSIC candidates."""
    p = {"SEED": int(seed), "MAX_CANDIDATES": CAND_K}
    p.update(params or {})
    res = lkh_solve(lkh_bin, xy=None if problem_file else xy, problem_file=problem_file, ew=ew,
                    params=p, workroot=workroot, want_cands=True, want_pi=True)
    n = len(xy)
    if not is_valid_tour(res["tour"], n):
        raise RuntimeError(f"invalid tour (rc={res['returncode']}): {res['stdout'][-600:]}")
    cost = tour_length(xy, res["tour"], ew)
    if res.get("cost_min") is not None and abs(cost - res["cost_min"]) > 0.5:
        raise RuntimeError(f"cost mismatch {cost} vs {res['cost_min']}")
    xy_store = np.asarray(xy, dtype=np.int32) if np.issubdtype(np.asarray(xy).dtype, np.integer) \
        else np.asarray(xy, dtype=np.float64)
    rec = {"n": n, "xy": xy_store, "tour": res["tour"].astype(np.int32), "cost": cost,
           "lower_bound": res["lower_bound"], "ascent_time": res["ascent_time"],
           "pre_time": res["pre_time"], "wall": res["wall"], "seed": int(seed),
           "dist": dist or "", "name": name or "", "ew": ew}
    if require_files and ("cand" not in res or "pi" not in res):
        raise RuntimeError("LKH did not write CANDIDATE_FILE/PI_FILE: " + res["stdout"][-600:])
    if "cand" in res:
        rec["cand"] = pad_cols(res["cand"]["cand"], CAND_K).astype(np.int32)
        rec["cand_alpha"] = pad_cols(res["cand"]["alpha"], CAND_K).astype(np.int64)
    if "pi" in res:
        rec["pi"] = res["pi"].astype(np.int64)
    if ensemble_M:
        ens = task_ensemble(lkh_bin, workroot, ensemble_M, seed0=int(seed) * 1000 + 17, xy=xy,
                            problem_file=problem_file, ew=ew)
        rec["ens_succ"] = ens["succ"]
        rec["ens_cost"] = ens["costs"].astype(np.float64)
        rec["ens_wall"] = float(ens["walls"].sum())
        rec["pop_cand"] = ens["cand"]
    return rec


def task_train_record(lkh_bin, xy_int, seed, workroot, params=None, dist=None):
    return task_record(lkh_bin, workroot, seed, xy=xy_int, params=params, dist=dist)


def task_candidates(lkh_bin, workroot, xy=None, problem_file=None, ew="EUC_2D", ctype="ALPHA",
                    subgradient=False, k=CAND_K, seed=1):
    p = {"CANDIDATE_SET_TYPE": ctype, "SUBGRADIENT": "YES" if subgradient else "NO",
         "MAX_CANDIDATES": k, "MAX_TRIALS": 1, "SEED": seed}
    res = lkh_solve(lkh_bin, xy=xy, problem_file=problem_file, ew=ew, params=p, workroot=workroot,
                    want_cands=True)
    if "cand" not in res:
        raise RuntimeError(f"no candidate file written ({ctype}): {res['stdout'][-800:]}")
    return {"cand": pad_cols(res["cand"]["cand"], k), "pre_time": res["pre_time"],
            "wall": res["wall"], "cand_stats": res["cand_stats"]}


def task_anytime(lkh_bin, workroot, budget, seed, xy=None, problem_file=None, ew="EUC_2D",
                 params=None, cand_file=None, pi_file=None, timeout=None):
    """timeout: wall-clock cap in seconds (LKH is stopped there; the trace printed so far is kept). LKH's own
    TIME_LIMIT does not count preprocessing, so a run with a long ascent needs the cap to stay within budget."""
    p = {"TIME_LIMIT": float(budget), "MAX_TRIALS": 100_000_000, "SEED": int(seed)}
    p.update(params or {})
    res = lkh_solve(lkh_bin, xy=xy, problem_file=problem_file, ew=ew, params=p, workroot=workroot,
                    cand_file=cand_file, pi_file=pi_file, timeout=budget * 3 + 600 if timeout is None else timeout)
    return {k: res.get(k) for k in ("events", "pre_time", "ascent_time", "lower_bound", "wall",
                                    "cost_min", "time_total", "cand_stats", "returncode",
                                    "timed_out", "t_start")}


# ------------------------------------------------------------ tour ensembles (TourFold)
def task_ensemble(lkh_bin, workroot, M, seed0, xy, problem_file=None, ew="EUC_2D", ctype="POPMUSIC",
                  k=CAND_K, extra_params=None):
    """M independent single-trial LKH tours that share one ascent-free candidate set.
    Run 0 builds the candidate set (POPMUSIC by default) and writes it; runs 1..M-1 read it together with
    zero penalties, so no run pays for a subgradient ascent. Returns successor arrays (n, M), tour costs,
    per-run wall times (sequential, one core), the shared candidate set and the best tour."""
    os.makedirs(workroot or tempfile.gettempdir(), exist_ok=True)
    wd = tempfile.mkdtemp(dir=workroot, prefix="ens_")
    try:
        n = len(xy)
        if problem_file is None:
            problem_file = os.path.join(wd, "p.tsp")
            write_tsp(problem_file, xy, ew=ew)
        cand_path, pi_path = os.path.join(wd, "c.cand"), os.path.join(wd, "zero.pi")
        base = {"MAX_TRIALS": 1, "SUBGRADIENT": "NO", "MAX_CANDIDATES": k}
        base.update(extra_params or {})
        tours, costs, walls = [], [], []
        cand = None
        for m in range(M):
            p = dict(base, SEED=int(seed0) + m)
            if m == 0:
                p.update(CANDIDATE_SET_TYPE=ctype, CANDIDATE_FILE=cand_path)
            else:
                p.update(CANDIDATE_FILE=cand_path, PI_FILE=pi_path)
            res = run_lkh(lkh_bin, problem_file, p, os.path.join(wd, f"r{m}"))
            if not is_valid_tour(res["tour"], n):
                raise RuntimeError(f"ensemble run {m} gave no valid tour: {res['stdout'][-500:]}")
            if m == 0:
                if not os.path.exists(cand_path):
                    raise RuntimeError("LKH did not write the ensemble candidate file: " + res["stdout"][-500:])
                cand = read_candidates(cand_path, k_pad=k)
                write_pi(pi_path, np.zeros(n))
            tours.append(res["tour"])
            costs.append(tour_length(xy, res["tour"], ew))
            walls.append(res["wall"])
        succ = np.stack([tour_succ_pred(t)[0] for t in tours], 1).astype(np.int32)
        b = int(np.argmin(costs))
        return {"succ": succ, "costs": np.array(costs), "walls": np.array(walls),
                "cand": pad_cols(cand["cand"], k).astype(np.int32), "best": tours[b].astype(np.int32),
                "best_cost": float(costs[b])}
    finally:
        shutil.rmtree(wd, ignore_errors=True)


def task_first_trial(lkh_bin, workroot, seed, xy, problem_file=None, ew="EUC_2D", ctype="POPMUSIC", k=CAND_K,
                     timeout=None):
    """The first trial of an ascent-free LKH run: candidate generation (POPMUSIC by default) and one trial.
    With the same seed it is the first trial of the corresponding full run, so a multi-start search can pause
    here, read its tours as a TourFold ensemble and resume. Returns the tour, its length, the run's own candidate
    lists (n, k) and the trace (preprocessing time, improvement events, wall time)."""
    p = {"CANDIDATE_SET_TYPE": ctype, "SUBGRADIENT": "NO", "MAX_TRIALS": 1, "MAX_CANDIDATES": int(k),
         "SEED": int(seed)}
    res = lkh_solve(lkh_bin, xy=None if problem_file else xy, problem_file=problem_file, ew=ew, params=p,
                    workroot=workroot, want_cands=True, timeout=timeout)
    n = len(xy)
    if not is_valid_tour(res["tour"], n):
        raise RuntimeError(f"first trial gave no valid tour (rc={res['returncode']}): {res['stdout'][-500:]}")
    if "cand" not in res:
        raise RuntimeError("LKH did not write the candidate file: " + res["stdout"][-500:])
    return {"tour": res["tour"].astype(np.int32), "cost": float(tour_length(xy, res["tour"], ew)),
            "cand": pad_cols(res["cand"]["cand"], k).astype(np.int32), "events": res["events"],
            "pre_time": res["pre_time"], "wall": res["wall"], "cost_min": res["cost_min"],
            "t_start": res["t_start"]}


def worker_ping(seconds=0.0):
    """Starts a pool worker (and its imports) ahead of timed work; returns the worker's process id."""
    time.sleep(seconds)
    return os.getpid()


def ensemble_pred(succ):
    n, M = succ.shape
    pred = np.empty_like(succ)
    ar = np.arange(n, dtype=succ.dtype)
    for m in range(M):
        pred[succ[:, m], m] = ar
    return pred


def ensemble_frequency(nb, succ):
    """freq[i, a] = share of ensemble tours that use edge (i, nb[i, a]) in either direction."""
    pred = ensemble_pred(succ)
    hit = (nb[:, :, None] == succ[:, None, :]) | (nb[:, :, None] == pred[:, None, :])
    return hit.mean(-1)


def frequency_candidates(nb, d, freq, k=CAND_K):
    """'Count' baseline: rank each node's kNN edges by ensemble frequency, ties broken by distance."""
    order = np.lexsort((d, -freq), axis=1)
    return np.take_along_axis(nb, order[:, :k], 1)


def mix_candidates(first, rest, k_first=3, k=CAND_K, fill=None):
    """Mixed candidate lists. Per node: the first k_first entries of `first`, then the entries of `rest` (in
    their order) that are not in the list yet, then - if `fill` is given - entries of `fill` not in the list yet,
    until k entries. Inputs are (n, *) arrays of 0-based node ids, -1 padded; a node never lists itself.
    Returns (n, k), -1 padded where fewer than k distinct neighbours are available."""
    first = np.asarray(first, dtype=np.int64)
    n = len(first)
    parts = [first[:, :k_first], np.asarray(rest, dtype=np.int64).reshape(n, -1)]
    if fill is not None:
        parts.append(np.asarray(fill, dtype=np.int64).reshape(n, -1))
    L = np.concatenate(parts, 1)
    if L.shape[1] < k:
        L = np.concatenate([L, np.full((n, k - L.shape[1]), -1, dtype=np.int64)], 1)
    W = L.shape[1]
    dup = ((L[:, :, None] == L[:, None, :]) & np.tri(W, W, -1, dtype=bool)[None]).any(2)   # seen earlier in row
    valid = (L >= 0) & ~dup & (L != np.arange(n)[:, None])
    order = np.argsort(~valid, axis=1, kind="stable")[:, :k]
    return np.where(np.take_along_axis(valid, order, 1), np.take_along_axis(L, order, 1), -1)


def fixing_stats(nb, score, tour, thresholds):
    """Precision / recall of 'fix edge if score >= t' against a reference tour, per threshold.
    Edges are undirected: an edge counts once and is fixed if either direction passes."""
    n, K = nb.shape
    lab = tour_edge_labels(nb, tour)
    i = np.repeat(np.arange(n), K)
    j = nb.reshape(-1)
    key = np.minimum(i, j) * n + np.maximum(i, j)
    s, y = score.reshape(-1), lab.reshape(-1)
    uniq, inv = np.unique(key, return_inverse=True)
    best = np.full(len(uniq), -np.inf)
    np.maximum.at(best, inv, s)
    is_tour = np.zeros(len(uniq), dtype=bool)
    is_tour[inv[y]] = True
    out = []
    for t in thresholds:
        sel = best >= t
        tp, nf = int((sel & is_tour).sum()), int(sel.sum())
        out.append({"t": float(t), "fixed": nf, "tp": tp, "n": int(n),
                    "precision": tp / nf if nf else float("nan"), "recall": tp / n})
    return out


# ------------------------------------------------------------ guided LKH runs (TourFold evaluation)
def select_fixed_edges(nb, prob, tau):
    """Greedy, consistent set of edges to fix: highest probability first, degree <= 2, no cycles.
    prob[i, a] is the probability of edge (i, nb[i, a]); both directions are merged by max."""
    n, K = nb.shape
    i = np.repeat(np.arange(n), K)
    j = nb.reshape(-1)
    p = prob.reshape(-1)
    lo, hi = np.minimum(i, j), np.maximum(i, j)
    key = lo * n + hi
    uniq, inv = np.unique(key, return_inverse=True)
    best = np.full(len(uniq), -np.inf)
    np.maximum.at(best, inv, p)
    order = np.argsort(-best, kind="stable")
    parent = np.arange(n)

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a
    deg = np.zeros(n, dtype=np.int64)
    out = []
    for e in order:
        if best[e] < tau or len(out) >= n - 1:
            break
        a, c = int(uniq[e] // n), int(uniq[e] % n)
        if deg[a] >= 2 or deg[c] >= 2:
            continue
        ra, rc = find(a), find(c)
        if ra == rc:
            continue
        parent[ra] = rc
        deg[a] += 1
        deg[c] += 1
        out.append((a, c))
    return np.array(out, dtype=np.int64).reshape(-1, 2)


def add_fixed_edges(src_problem, dst_problem, edges):
    """Copy a TSPLIB problem file and append a FIXED_EDGES_SECTION (0-based edges in, 1-based out)."""
    with _open_text(src_problem) as f:
        lines = [ln.rstrip("\n") for ln in f]
    while lines and lines[-1].strip() in ("", "EOF"):
        lines.pop()
    body = "\n".join(f"{a + 1} {b + 1}" for a, b in np.asarray(edges).tolist())
    with open(dst_problem, "w") as f:
        f.write("\n".join(lines) + "\nFIXED_EDGES_SECTION\n" + (body + "\n" if body else "") + "-1\nEOF\n")


def task_guided(lkh_bin, workroot, budget, seed, xy, cand, pi=None, init_tour=None, problem_file=None,
                ew="EUC_2D", fixed_edges=None, alpha=None, extra_params=None, return_tour=False, timeout=None):
    """LKH with externally supplied candidates (+ penalties, initial tour, fixed edges); no ascent."""
    os.makedirs(workroot or tempfile.gettempdir(), exist_ok=True)
    wd = tempfile.mkdtemp(dir=workroot, prefix="g_")
    try:
        n = len(xy)
        base = problem_file
        if base is None:
            base = os.path.join(wd, "p.tsp")
            write_tsp(base, xy, ew=ew)
        pf = base
        if fixed_edges is not None and len(fixed_edges):
            pf = os.path.join(wd, "fixed.tsp")
            add_fixed_edges(base, pf, fixed_edges)
        cf, pif = os.path.join(wd, "g.cand"), os.path.join(wd, "g.pi")
        write_candidates(cf, cand, alpha)
        write_pi(pif, np.zeros(n) if pi is None else pi)
        params = {"SUBGRADIENT": "NO", "TIME_LIMIT": float(budget), "MAX_TRIALS": 100_000_000,
                  "SEED": int(seed), "MAX_CANDIDATES": int(np.asarray(cand).shape[1])}
        if init_tour is not None:
            tp = os.path.join(wd, "init.tour")
            write_tour(tp, init_tour)
            params["INITIAL_TOUR_FILE"] = tp
        params.update(extra_params or {})
        params["CANDIDATE_FILE"], params["PI_FILE"] = cf, pif
        res = run_lkh(lkh_bin, pf, params, os.path.join(wd, "run"),
                      timeout=budget * 3 + 600 if timeout is None else timeout)
        ok = is_valid_tour(res["tour"], n)
        cost = tour_length(xy, res["tour"], ew) if ok else None
        out = {k: res.get(k) for k in ("events", "pre_time", "wall", "cost_min", "returncode", "timed_out",
                                       "t_start")}
        out["cost_check"] = cost
        if fixed_edges is not None and len(fixed_edges):
            # LKH gives fixed edges length 0, so its reported costs miss their total length. Every tour of the
            # run contains all fixed edges, hence one constant shift restores true lengths for all events.
            fe = np.asarray(fixed_edges, dtype=np.int64).reshape(-1, 2)
            xa = np.asarray(xy)
            out["fixed_len"] = float(edge_len(xa[fe[:, 0]], xa[fe[:, 1]], ew).sum())
            if cost is not None and res.get("cost_min") is not None:
                shift = float(cost - res["cost_min"])
            else:
                shift = out["fixed_len"]
            out["cost_shift"] = shift
            out["events"] = [(i, int(round(c + shift)), t) for i, c, t in (res.get("events") or [])]
            if res.get("cost_min") is not None:
                out["cost_min"] = int(round(res["cost_min"] + shift))
        if not ok:
            out["stdout_tail"] = (res.get("stdout") or "")[-800:]
        if return_tour:
            out["tour"] = res["tour"] if ok else None
        return out
    finally:
        shutil.rmtree(wd, ignore_errors=True)


def abs_events(res, offset=0.0):
    """(cost, absolute time) pairs: offset + LKH preprocessing + time inside the run."""
    pre = (res.get("pre_time") or 0.0) + offset
    return [(int(c), pre + float(t)) for _, c, t in (res.get("events") or [])]


def curve_abs(events, ref, budgets):
    ev = sorted(events, key=lambda e: e[1])
    out = []
    for b in budgets:
        cs = [c for c, t in ev if t <= b]
        out.append(gap_pct(min(cs), ref) if cs else np.nan)
    return out


def time_to_gap_abs(events, ref, target_pct):
    for c, t in sorted(events, key=lambda e: e[1]):
        if gap_pct(c, ref) <= target_pct + 1e-12:
            return t
    return float("inf")
''', "tsp_model.py": r'''"""
tsp_model - AlphaFold-inspired sparse pair model ("PairFormer-lite") that predicts
LKH candidate edges and node penalties (pi) for Euclidean TSP.

Representation (as in AlphaFold's Evoformer):
  * single (node) representation  h_i            (N, dn)
  * pair (edge) representation    z_ij on kNN    (N, K, de), directed edges i -> NB[i, a]
Block = pair-biased node attention -> node transition -> node->edge product update ->
        reverse-edge mixing -> triangle multiplicative update (outgoing, incoming) ->
        [optional triangle attention] -> edge transition.
Triangles are the directed kNN triangles {i->j, i->k, j->k}; the same list serves the
"outgoing" (z_ij += sum_k a_ik * b_jk) and "incoming" (z_jk += sum_i a_ij * b_ik) updates.
All input features are invariant to translation, rotation, scale and node order.
Heads: edge logits (top-k -> CANDIDATE_FILE) and scale-equivariant node penalties
pi_i = s_i * f_i (-> PI_FILE). Optional AlphaFold-style recycling.
"""
from __future__ import annotations

import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

PRECISION = 100  # LKH default


# ------------------------------------------------------------------ graph + features
def knn_np(xy, K):
    from scipy.spatial import cKDTree
    n = len(xy)
    d, idx = cKDTree(xy).query(xy, k=K + 1)
    self_mask = idx == np.arange(n)[:, None]
    no_self = ~self_mask.any(1)
    self_mask[no_self, -1] = True
    keep = ~self_mask
    return idx[keep].reshape(n, K), d[keep].reshape(n, K)


def extend_graph(xyn, NB, ens_succ, extra):
    """TourFold v2 graph: kNN-K plus up to `extra` neighbours that the tour ensemble uses but kNN-K lacks
    (long edges, e.g. between clusters). They replace the farthest kNN slots, so every row still has K entries;
    the most frequently used ensemble edges come first. Rows are re-sorted by distance.
    Returns NB (n,K), D (n,K) and a flag (n,K) marking the ensemble-only edges."""
    n, K = NB.shape
    succ = np.asarray(ens_succ, dtype=np.int64)
    Mt = succ.shape[1]
    pred = np.empty_like(succ)
    ar = np.arange(n)
    for m in range(Mt):
        pred[succ[:, m], m] = ar
    cand = np.concatenate([succ, pred], 1)                              # (n, 2M) ensemble neighbours
    cand = np.where((cand[:, :, None] == NB[:, None, :]).any(-1), -1, cand)
    S = np.sort(cand, 1)
    first = np.ones_like(S, dtype=bool)
    first[:, 1:] = S[:, 1:] != S[:, :-1]
    valid = (S >= 0) & first
    cnt = (S[:, :, None] == S[:, None, :]).sum(-1)
    dist = np.linalg.norm(xyn[np.maximum(S, 0)] - xyn[:, None, :], axis=-1)
    score = np.where(valid, cnt * 10.0 - dist, -np.inf)                 # frequency first, then distance
    order = np.argsort(-score, 1, kind="stable")[:, :extra]
    chosen = np.take_along_axis(S, order, 1)
    nv = np.isfinite(np.take_along_axis(score, order, 1)).sum(1)       # ensemble-only neighbours used
    pos = np.arange(K)[None, :]
    keep = pos < (K - nv)[:, None]
    tail = np.take_along_axis(chosen, np.clip(pos - (K - nv)[:, None], 0, extra - 1), 1)
    NB2 = np.where(keep, NB, tail)
    D2 = np.linalg.norm(xyn[NB2] - xyn[:, None, :], axis=-1)
    o = np.argsort(D2, 1, kind="stable")
    return (np.take_along_axis(NB2, o, 1), np.take_along_axis(D2, o, 1),
            np.take_along_axis(~keep, o, 1))


def prepare_instance(xy, K, tour=None, pi=None, precision=PRECISION, ens_succ=None, ens_cost=None, extra=0):
    """Normalize coordinates, build kNN graph, labels and pi targets (numpy).
    extra > 0 (TourFold v2): the graph also gets up to `extra` ensemble edges per node (see extend_graph)."""
    xy = np.asarray(xy, dtype=np.float64)
    n = len(xy)
    if n <= K:
        raise ValueError(f"n={n} must exceed K={K}")
    mn = xy.min(0)
    span = float((xy.max(0) - mn).max()) or 1.0
    xyn = (xy - mn) / span
    NB, D = knn_np(xyn, K)
    s = np.maximum(D[:, : max(1, K // 2)].mean(1), 1e-9)
    ens_only = None
    if extra and ens_succ is not None:
        NB, D, ens_only = extend_graph(xyn, NB, ens_succ, int(extra))
    out = {"n": n, "span": span, "xy": xyn.astype(np.float32), "NB": NB.astype(np.int64),
           "D": D.astype(np.float32), "s": s.astype(np.float32),
           "s_glob": np.full(n, np.median(s), dtype=np.float32)}
    if ens_only is not None:
        out["ens_only"] = ens_only.astype(np.float32)
    if tour is not None:
        t = np.asarray(tour)
        succ = np.empty(n, np.int64)
        pred = np.empty(n, np.int64)
        succ[t] = np.roll(t, -1)
        pred[t] = np.roll(t, 1)
        out["y"] = (NB == succ[:, None]) | (NB == pred[:, None])
    if pi is not None:
        out["pi_t"] = (np.asarray(pi, dtype=np.float64) / (precision * span * s)).astype(np.float32)
    if ens_succ is not None:
        out.update(ensemble_features(xyn, NB, s, ens_succ, ens_cost))
    return out


def ensemble_features(xyn, NB, s, ens_succ, ens_cost=None):
    """TourFold 'tour alignment' (the MSA analogue): per tour m and node i, how tour m connects i.
    Everything is symmetric in the two tour neighbours, so a reversed tour gives the same features."""
    succ = np.asarray(ens_succ, dtype=np.int64)
    n, M = succ.shape
    K = NB.shape[1]
    pred = np.empty_like(succ)
    ar = np.arange(n)
    for m in range(M):
        pred[succ[:, m], m] = ar
    eq_s = NB[:, :, None] == succ[:, None, :]
    eq_p = NB[:, :, None] == pred[:, None, :]
    pos_s = np.where(eq_s.any(1), eq_s.argmax(1), K)
    pos_p = np.where(eq_p.any(1), eq_p.argmax(1), K)
    ds = np.log(np.linalg.norm(xyn[succ] - xyn[:, None, :], axis=-1) / s[:, None] + 1e-4)
    dp = np.log(np.linalg.norm(xyn[pred] - xyn[:, None, :], axis=-1) / s[:, None] + 1e-4)
    fs = ((succ[:, None, :] == succ[:, :, None]) | (pred[:, None, :] == succ[:, :, None])).mean(-1)
    fp = ((succ[:, None, :] == pred[:, :, None]) | (pred[:, None, :] == pred[:, :, None])).mean(-1)
    cost = np.ones(M) if ens_cost is None else np.asarray(ens_cost, dtype=np.float64)
    q = (cost - cost.min()) / max(cost.mean(), 1e-9) * 100.0
    msa_f = np.stack([np.minimum(ds, dp), np.maximum(ds, dp), np.broadcast_to(q[None, :], (n, M)),
                      np.minimum(fs, fp), np.maximum(fs, fp)], -1)
    hit = eq_s | eq_p
    top = np.argsort(cost, kind="stable")[: max(1, M // 2)]
    ef_ens = np.stack([hit.mean(-1), hit[:, :, top].mean(-1)], -1)
    return {"msa_f": msa_f.transpose(1, 0, 2).astype(np.float32),
            "msa_ps": pos_s.T.astype(np.int64), "msa_pp": pos_p.T.astype(np.int64),
            "ens_succ": succ.T.copy(), "ens_pred": pred.T.copy(),
            "ef_ens": ef_ens.astype(np.float32)}


def collate(items):
    """Concatenate prepared instances into one disjoint graph."""
    off, N = [], 0
    for it in items:
        off.append(N)
        N += it["n"]
    cat = lambda k: np.concatenate([it[k] for it in items], 0)  # noqa: E731
    b = {"xy": torch.from_numpy(cat("xy")), "D": torch.from_numpy(cat("D")),
         "s": torch.from_numpy(cat("s")), "s_glob": torch.from_numpy(cat("s_glob")),
         "NB": torch.from_numpy(np.concatenate([it["NB"] + o for it, o in zip(items, off)], 0)),
         "bidx": torch.from_numpy(np.concatenate([np.full(it["n"], i) for i, it in enumerate(items)])),
         "offsets": torch.tensor(off + [N]), "span": torch.tensor([it["span"] for it in items]),
         "B": len(items)}
    if all("y" in it for it in items):
        b["y"] = torch.from_numpy(cat("y"))
    if all("pi_t" in it for it in items):
        b["pi_t"] = torch.from_numpy(cat("pi_t"))
    if all("ens_only" in it for it in items):
        b["ens_only"] = torch.from_numpy(cat("ens_only"))
    if all("msa_f" in it for it in items):
        Ms = {it["msa_f"].shape[0] for it in items}
        if len(Ms) != 1:
            raise ValueError(f"all instances in a batch need the same ensemble size, got {Ms}")
        b["msa_f"] = torch.from_numpy(np.concatenate([it["msa_f"] for it in items], 1))
        b["msa_ps"] = torch.from_numpy(np.concatenate([it["msa_ps"] for it in items], 1))
        b["msa_pp"] = torch.from_numpy(np.concatenate([it["msa_pp"] for it in items], 1))
        b["ens_succ"] = torch.from_numpy(np.concatenate([it["ens_succ"] + o for it, o in zip(items, off)], 1))
        b["ens_pred"] = torch.from_numpy(np.concatenate([it["ens_pred"] + o for it, o in zip(items, off)], 1))
        b["ef_ens"] = torch.from_numpy(cat("ef_ens"))
    return b


def to_device(b, device):
    return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in b.items()}


@torch.no_grad()
def graph_context(b, k_tri=None, chunk_nodes=40000):
    """Invariant features, reverse-edge index and directed triangles (all on device).
    k_tri: the third node k of a triangle must be among the first k_tri neighbours of both i and j
    (fewer, more local triangles -> less memory)."""
    xy, NB, D, s, sg = b["xy"], b["NB"], b["D"], b["s"], b["s_glob"]
    N, K = NB.shape
    kt = K if not k_tri else min(int(k_tri), K)
    ar = torch.arange(N, device=NB.device)
    eq = NB[NB] == ar[:, None, None]                     # (N,K,K): NB[NB[i,a], c] == i
    mutual = eq.any(-1)
    rev_pos = eq.float().argmax(-1)
    rank = torch.arange(K, device=NB.device, dtype=xy.dtype).expand(N, K) / K
    rev_rank = torch.where(mutual, rev_pos.to(xy.dtype) / K, torch.full_like(rank, 1.5))
    vec = xy[NB] - xy[:, None, :]
    ang = torch.atan2(vec[..., 1], vec[..., 0])
    rel = ang - ang[:, :1]
    d1 = D / s[:, None]
    ef = torch.stack([d1, D / s[NB], D / sg[:, None], torch.log(d1 + 1e-4), rank, mutual.to(xy.dtype),
                      rev_rank, torch.cos(rel), torch.sin(rel).abs()], -1)
    if "ef_ens" in b:
        ef = torch.cat([ef, b["ef_ens"]], -1)
    if "ens_only" in b:
        ef = torch.cat([ef, b["ens_only"][..., None].to(ef.dtype)], -1)
    cen = xy[NB].mean(1)
    indeg = torch.bincount(NB.reshape(-1), minlength=N).to(xy.dtype) / K
    nf = torch.stack([s / sg, D[:, 0] / s, D[:, -1] / s, (cen - xy).norm(dim=-1) / s, indeg], -1)
    tris = []
    step = max(1, chunk_nodes // max(1, K * kt * kt // 4096))
    for st in range(0, N, step):
        nb = NB[st:st + step]
        M = NB[nb][:, :, None, :kt] == nb[:, None, :kt, None]   # (n,a,b,c): NB[NB[i,a],c] == NB[i,b]
        idx = M.nonzero()
        i = idx[:, 0] + st
        a, bb, c = idx[:, 1], idx[:, 2], idx[:, 3]
        tris.append(torch.stack([i * K + a, i * K + bb, NB[i, a] * K + c]))
    tri = torch.cat(tris, 1) if tris else torch.zeros(3, 0, dtype=torch.long, device=NB.device)
    return {"node_feat": nf, "edge_feat": ef, "mutual": mutual, "rev_pos": rev_pos, "tri": tri}


# ---------------------------------------------------------------------------- modules
class Transition(nn.Module):
    def __init__(self, d, mult=2):
        super().__init__()
        self.ln = nn.LayerNorm(d)
        self.l1 = nn.Linear(d, d * mult)
        self.l2 = nn.Linear(d * mult, d)

    def forward(self, x):
        return self.l2(F.gelu(self.l1(self.ln(x))))


class NodeAttention(nn.Module):
    """Attention of node i over its K neighbours, biased by the pair representation."""

    def __init__(self, dn, de, heads=4):
        super().__init__()
        self.h, self.dh = heads, dn // heads
        self.ln, self.lnz = nn.LayerNorm(dn), nn.LayerNorm(de)
        self.q, self.kv = nn.Linear(dn, dn), nn.Linear(dn, 2 * dn)
        self.bias = nn.Linear(de, heads)
        self.g, self.o = nn.Linear(dn, dn), nn.Linear(dn, dn)

    def forward(self, h, z, NB):
        N, K = NB.shape
        x = self.ln(h)
        q = self.q(x).view(N, self.h, self.dh)
        k, v = self.kv(x)[NB].split(self.h * self.dh, -1)
        k, v = k.view(N, K, self.h, self.dh), v.view(N, K, self.h, self.dh)
        logit = torch.einsum("nhd,nkhd->nhk", q, k) / math.sqrt(self.dh)
        logit = logit + self.bias(self.lnz(z)).permute(0, 2, 1)
        att = logit.float().softmax(-1).to(v.dtype)
        out = torch.einsum("nhk,nkhd->nhd", att, v).reshape(N, -1)
        return torch.sigmoid(self.g(x)) * self.o(out)


class EdgeFromNodes(nn.Module):
    def __init__(self, dn, de):
        super().__init__()
        self.ln = nn.LayerNorm(dn)
        self.l, self.r, self.o = nn.Linear(dn, de), nn.Linear(dn, de), nn.Linear(de, de)

    def forward(self, h, NB):
        x = self.ln(h)
        return self.o(self.l(x)[:, None, :] * self.r(x)[NB])


class ReverseMix(nn.Module):
    def __init__(self, de):
        super().__init__()
        self.ln, self.l = nn.LayerNorm(de), nn.Linear(de, de)

    def forward(self, z, NB, rev_pos, mutual):
        zr = z[NB, rev_pos]                                    # z_ji for edge i->j
        return self.l(self.ln(zr)) * mutual[..., None].to(z.dtype)


class TriangleMultiplication(nn.Module):
    """AlphaFold Alg. 11/12 on sparse directed kNN triangles."""

    def __init__(self, de, dh, mode):
        super().__init__()
        self.mode = mode
        self.ln = nn.LayerNorm(de)
        self.a, self.ag = nn.Linear(de, dh), nn.Linear(de, dh)
        self.b, self.bg = nn.Linear(de, dh), nn.Linear(de, dh)
        self.g = nn.Linear(de, de)
        self.ln_o, self.o = nn.LayerNorm(dh), nn.Linear(dh, de)

    def forward(self, z, tri):
        N, K, de = z.shape
        x = self.ln(z).reshape(N * K, de)
        a = torch.sigmoid(self.ag(x)) * self.a(x)
        b = torch.sigmoid(self.bg(x)) * self.b(x)
        e_ij, e_ik, e_jk = tri
        tgt, s1, s2 = (e_ij, e_ik, e_jk) if self.mode == "out" else (e_jk, e_ij, e_ik)
        prod = (a[s1] * b[s2]).float()
        m = torch.zeros(N * K, prod.shape[1], device=z.device, dtype=torch.float32).index_add_(0, tgt, prod)
        cnt = torch.bincount(tgt, minlength=N * K).clamp_min(1).float().sqrt()
        m = (m / cnt[:, None]).to(x.dtype)
        out = torch.sigmoid(self.g(x)) * self.o(self.ln_o(m))
        return out.view(N, K, de)


class TriangleAttention(nn.Module):
    """AlphaFold Alg. 13 (starting node) on sparse triangles: edge i->j attends over k with
    keys/values from i->k and a bias from j->k."""

    def __init__(self, de, heads=4):
        super().__init__()
        self.h = heads
        self.dh = max(8, de // heads)
        self.ln = nn.LayerNorm(de)
        self.q, self.k, self.v = (nn.Linear(de, self.h * self.dh) for _ in range(3))
        self.bias, self.g = nn.Linear(de, heads), nn.Linear(de, self.h * self.dh)
        self.o = nn.Linear(self.h * self.dh, de)

    def forward(self, z, tri):
        N, K, de = z.shape
        E = N * K
        x = self.ln(z).reshape(E, de)
        e_ij, e_ik, e_jk = tri
        q = self.q(x)[e_ij].view(-1, self.h, self.dh)
        k = self.k(x)[e_ik].view(-1, self.h, self.dh)
        logit = ((q * k).sum(-1) / math.sqrt(self.dh) + self.bias(x)[e_jk]).float()   # (T,h)
        mx = torch.full((E, self.h), -1e30, device=z.device).scatter_reduce(
            0, e_ij[:, None].expand(-1, self.h), logit, reduce="amax", include_self=True)
        w = torch.exp(logit - mx[e_ij])
        den = torch.zeros(E, self.h, device=z.device).index_add_(0, e_ij, w).clamp_min(1e-9)
        w = w / den[e_ij]
        v = self.v(x)[e_ik].view(-1, self.h, self.dh).float()
        out = torch.zeros(E, self.h, self.dh, device=z.device).index_add_(0, e_ij, w[..., None] * v)
        out = (torch.sigmoid(self.g(x)) * out.reshape(E, -1).to(x.dtype))
        return self.o(out).view(N, K, de)


class Block(nn.Module):
    def __init__(self, dn, de, dtri, heads, tri_attn):
        super().__init__()
        self.node_att = NodeAttention(dn, de, heads)
        self.node_tr = Transition(dn)
        self.e_from_n = EdgeFromNodes(dn, de)
        self.rev = ReverseMix(de)
        self.tri_out = TriangleMultiplication(de, dtri, "out")
        self.tri_in = TriangleMultiplication(de, dtri, "in")
        self.tri_att = TriangleAttention(de, heads) if tri_attn else None
        self.edge_tr = Transition(de)

    def forward(self, h, z, NB, rev_pos, mutual, tri):
        h = h + self.node_att(h, z, NB)
        h = h + self.node_tr(h)
        z = z + self.e_from_n(h, NB)
        z = z + self.rev(z, NB, rev_pos, mutual)
        z = z + self.tri_out(z, tri)
        z = z + self.tri_in(z, tri)
        if self.tri_att is not None:
            z = z + self.tri_att(z, tri)
        z = z + self.edge_tr(z)
        return h, z


class PairFormerTSP(nn.Module):
    def __init__(self, dn=128, de=64, dtri=32, layers=8, heads=4, tri_attn=False, recycles=0,
                 grad_ckpt=False, k_tri=12, node_in=5, edge_in=9):
        super().__init__()
        self.cfg = dict(dn=dn, de=de, dtri=dtri, layers=layers, heads=heads, tri_attn=tri_attn,
                        recycles=recycles, grad_ckpt=grad_ckpt, k_tri=k_tri, node_in=node_in, edge_in=edge_in)
        self.node_in = nn.Sequential(nn.Linear(node_in, dn), nn.GELU(), nn.Linear(dn, dn))
        self.edge_in = nn.Sequential(nn.Linear(edge_in, de), nn.GELU(), nn.Linear(de, de))
        self.rec_h = nn.Sequential(nn.LayerNorm(dn), nn.Linear(dn, dn))
        self.rec_z = nn.Sequential(nn.LayerNorm(de), nn.Linear(de, de))
        for m in (self.rec_h[1], self.rec_z[1]):
            nn.init.zeros_(m.weight)
            nn.init.zeros_(m.bias)
        self.blocks = nn.ModuleList(Block(dn, de, dtri, heads, tri_attn) for _ in range(layers))
        self.edge_head = nn.Sequential(nn.LayerNorm(de), nn.Linear(de, de), nn.GELU(), nn.Linear(de, 1))
        self.pi_head = nn.Sequential(nn.LayerNorm(dn), nn.Linear(dn, dn), nn.GELU(), nn.Linear(dn, 1))
        self.grad_ckpt = grad_ckpt
        self.recycles = recycles

    def trunk(self, ctx, NB, h_prev=None, z_prev=None):
        h = self.node_in(ctx["node_feat"])
        z = self.edge_in(ctx["edge_feat"])
        if h_prev is not None:
            h = h + self.rec_h(h_prev)
            z = z + self.rec_z(z_prev)
        for blk in self.blocks:
            if self.grad_ckpt and self.training:
                h, z = checkpoint(blk, h, z, NB, ctx["rev_pos"], ctx["mutual"], ctx["tri"], use_reentrant=False)
            else:
                h, z = blk(h, z, NB, ctx["rev_pos"], ctx["mutual"], ctx["tri"])
        return h, z

    def forward(self, ctx, NB, n_recycle=None):
        n_rec = self.recycles if n_recycle is None else n_recycle
        h_prev = z_prev = None
        if n_rec > 0:
            with torch.no_grad():
                for _ in range(n_rec):
                    h_prev, z_prev = self.trunk(ctx, NB, h_prev, z_prev)
                    h_prev, z_prev = h_prev.detach(), z_prev.detach()
        h, z = self.trunk(ctx, NB, h_prev, z_prev)
        logit = self.edge_head(z).squeeze(-1).float()                  # (N,K) directed
        mutual, rev_pos = ctx["mutual"], ctx["rev_pos"]
        logit_rev = logit[NB, rev_pos]
        sym = torch.where(mutual, 0.5 * (logit + logit_rev), logit)
        f = self.pi_head(h).squeeze(-1).float()
        return {"edge_logit": sym, "pi_f": f}


# ------------------------------------------------------------------------- losses
def scatter_mean(x, idx, B):
    s = torch.zeros(B, device=x.device, dtype=x.dtype).index_add_(0, idx, x)
    c = torch.zeros(B, device=x.device, dtype=x.dtype).index_add_(0, idx, torch.ones_like(x))
    return s / c.clamp_min(1)


def compute_losses(out, b, w_bce=0.5, w_pi=1.0):
    logit = out["edge_logit"]
    y = b["y"].float()
    K = y.shape[1]
    has = y.sum(1) > 0
    logp = F.log_softmax(logit, 1)
    ce = -((y / y.sum(1, keepdim=True).clamp_min(1)) * logp).sum(1)[has].mean()
    bce = F.binary_cross_entropy_with_logits(logit, y, pos_weight=torch.tensor((K - 2) / 2, device=y.device))
    loss = ce + w_bce * bce
    logs = {"ce": ce.item(), "bce": bce.item()}
    if "pi_t" in b and w_pi > 0:
        s, bidx, B = b["s"], b["bidx"], b["B"]
        pi_hat, pi_tgt = s * out["pi_f"], s * b["pi_t"]
        d = (pi_hat - scatter_mean(pi_hat, bidx, B)[bidx]) - (pi_tgt - scatter_mean(pi_tgt, bidx, B)[bidx])
        mse = (d / s).pow(2).mean()
        loss = loss + w_pi * mse
        logs["pi_mse"] = mse.item()
    logs["loss"] = loss.item()
    return loss, logs


# ---------------------------------------------------------------------- inference
def autocast_dtype(device):
    if device.type == "cuda" and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return None


@torch.no_grad()
def predict(model, xy, K, device, k_out=5, n_recycle=None):
    """Return LKH-ready candidates (n,k_out), alpha ranks, integer pi and timings."""
    t0 = time.perf_counter()
    prep = prepare_instance(xy, K)
    b = to_device(collate([prep]), device)
    t1 = time.perf_counter()
    ctx = graph_context(b, k_tri=model.cfg.get("k_tri"))
    dt = autocast_dtype(device)
    model.eval()
    with torch.autocast(device_type=device.type, dtype=dt, enabled=dt is not None):
        out = model(ctx, b["NB"], n_recycle=n_recycle)
    logit = out["edge_logit"].float()
    top = logit.topk(k_out, dim=1).indices
    cand = torch.gather(b["NB"], 1, top).cpu().numpy()
    pi_hat = (b["s"] * out["pi_f"]).double()
    pi_hat = pi_hat - pi_hat.mean()
    pi_int = torch.round(pi_hat * PRECISION * prep["span"]).long().cpu().numpy()
    if device.type == "cuda":
        torch.cuda.synchronize()
    t2 = time.perf_counter()
    alpha = np.tile(np.arange(k_out, dtype=np.int64) * 100, (len(cand), 1))
    return {"cand": cand, "alpha": alpha, "pi": pi_int, "logit": logit.cpu().numpy(),
            "NB": prep["NB"], "t_prep": t1 - t0, "t_model": t2 - t1, "t_total": t2 - t0}


# =============================================================================== TourFold
# The ensemble of cheap local-optimum tours plays the role of AlphaFold's MSA:
#   column attention  = for one node, compare how the M tours connect it
#   tour convolution  = context along each tour (successor + predecessor, shared weights)
#   row attention     = for one tour and node, look at the node's kNN neighbours, biased by the pair rep
#   outer-product mean= co-variation across tours  ->  pair (edge) representation
# followed by the pair stack with triangle updates (Block above).

class MSAColumnAttention(nn.Module):
    def __init__(self, dm, heads=4):
        super().__init__()
        self.h, self.dh = heads, max(4, dm // heads)
        self.ln = nn.LayerNorm(dm)
        self.qkv = nn.Linear(dm, 3 * self.h * self.dh)
        self.g, self.o = nn.Linear(dm, self.h * self.dh), nn.Linear(self.h * self.dh, dm)

    def forward(self, msa):
        M, N, _ = msa.shape
        x = self.ln(msa)
        q, k, v = self.qkv(x).view(M, N, 3, self.h, self.dh).unbind(2)
        logit = torch.einsum("mnhd,lnhd->nhml", q, k) / math.sqrt(self.dh)
        att = logit.float().softmax(-1).to(v.dtype)
        out = torch.einsum("nhml,lnhd->mnhd", att, v).reshape(M, N, -1)
        return self.o(torch.sigmoid(self.g(x)) * out)


class MSATourConv(nn.Module):
    """msa[m,i] += f(msa[m,i], msa[m,succ_m(i)] + msa[m,pred_m(i)]) - direction-free context along each tour."""

    def __init__(self, dm):
        super().__init__()
        self.ln = nn.LayerNorm(dm)
        self.self_w, self.nbr_w = nn.Linear(dm, dm), nn.Linear(dm, dm)
        self.o = nn.Linear(dm, dm)

    def forward(self, msa, succ, pred):
        M, N, dm = msa.shape
        x = self.ln(msa)
        nb = torch.gather(x, 1, succ[..., None].expand(M, N, dm)) + torch.gather(x, 1, pred[..., None].expand(M, N, dm))
        return self.o(F.gelu(self.self_w(x) + self.nbr_w(nb)))


class MSARowAttention(nn.Module):
    """For tour m and node i: attention over i's kNN neighbours j, biased by the pair representation z_ij."""

    def __init__(self, dm, de, heads=4):
        super().__init__()
        self.h, self.dh = heads, max(4, dm // heads)
        self.ln, self.lnz = nn.LayerNorm(dm), nn.LayerNorm(de)
        self.q, self.kv = nn.Linear(dm, self.h * self.dh), nn.Linear(dm, 2 * self.h * self.dh)
        self.bias = nn.Linear(de, heads)
        self.g, self.o = nn.Linear(dm, self.h * self.dh), nn.Linear(self.h * self.dh, dm)

    def forward(self, msa, z, NB):
        M, N, _ = msa.shape
        K = NB.shape[1]
        x = self.ln(msa)
        q = self.q(x).view(M, N, self.h, self.dh)
        k, v = self.kv(x)[:, NB].split(self.h * self.dh, -1)               # (M,N,K,h*dh)
        k, v = k.view(M, N, K, self.h, self.dh), v.view(M, N, K, self.h, self.dh)
        logit = torch.einsum("mnhd,mnkhd->mnhk", q, k) / math.sqrt(self.dh)
        logit = logit + self.bias(self.lnz(z)).permute(0, 2, 1)[None]
        att = logit.float().softmax(-1).to(v.dtype)
        out = torch.einsum("mnhk,mnkhd->mnhd", att, v).reshape(M, N, -1)
        return self.o(torch.sigmoid(self.g(x)) * out)


class OuterProductMean(nn.Module):
    """z_ij += W * mean_m (a[m,i] (x) b[m,j]) on kNN edges: co-variation of how the tours treat i and j."""

    def __init__(self, dm, de, c=8):
        super().__init__()
        self.c = c
        self.ln = nn.LayerNorm(dm)
        self.a, self.b = nn.Linear(dm, c), nn.Linear(dm, c)
        self.o = nn.Linear(c * c, de)

    def forward(self, msa, NB):
        M, N, _ = msa.shape
        K = NB.shape[1]
        x = self.ln(msa)
        a, b = self.a(x), self.b(x)[:, NB]                                     # (M,N,c), (M,N,K,c)
        opm = torch.einsum("mnc,mnkd->nkcd", a.float(), b.float()) / M
        return self.o(opm.reshape(N, K, self.c * self.c).to(x.dtype))


class EvoBlock(nn.Module):
    def __init__(self, dn, de, dm, dtri, heads, tri_attn, opm_c=8):
        super().__init__()
        self.col = MSAColumnAttention(dm, heads)
        self.conv = MSATourConv(dm)
        self.row = MSARowAttention(dm, de, heads)
        self.msa_tr = Transition(dm)
        self.opm = OuterProductMean(dm, de, opm_c)
        self.pair = Block(dn, de, dtri, heads, tri_attn)

    def forward(self, msa, h, z, NB, rev_pos, mutual, tri, succ, pred):
        msa = msa + self.col(msa)
        msa = msa + self.conv(msa, succ, pred)
        msa = msa + self.row(msa, z, NB)
        msa = msa + self.msa_tr(msa)
        z = z + self.opm(msa, NB)
        h, z = self.pair(h, z, NB, rev_pos, mutual, tri)
        return msa, h, z


class TourFoldNet(nn.Module):
    """use_ensemble=False gives the single-instance ablation (same pair stack, no tour alignment)."""

    def __init__(self, dn=128, de=64, dm=32, dtri=32, msa_layers=2, pair_layers=6, heads=4,
                 tri_attn=False, use_ensemble=True, k_tri=12, K=16, opm_c=8, grad_ckpt=False,
                 node_in=5, edge_in=9, msa_in=5, graph_extra=0):
        super().__init__()
        self.cfg = dict(dn=dn, de=de, dm=dm, dtri=dtri, msa_layers=msa_layers, pair_layers=pair_layers,
                        heads=heads, tri_attn=tri_attn, use_ensemble=use_ensemble, k_tri=k_tri, K=K,
                        opm_c=opm_c, grad_ckpt=grad_ckpt, node_in=node_in, edge_in=edge_in, msa_in=msa_in,
                        graph_extra=int(graph_extra if use_ensemble else 0))
        self.use_ensemble = use_ensemble
        self.node_in = nn.Sequential(nn.Linear(node_in, dn), nn.GELU(), nn.Linear(dn, dn))
        e_in = edge_in + (2 if use_ensemble else 0) + (1 if (use_ensemble and graph_extra) else 0)
        self.edge_in = nn.Sequential(nn.Linear(e_in, de), nn.GELU(),
                                     nn.Linear(de, de))
        if use_ensemble:
            self.msa_in = nn.Sequential(nn.Linear(msa_in, dm), nn.GELU(), nn.Linear(dm, dm))
            self.pos_emb = nn.Embedding(K + 1, dm)
            self.evo = nn.ModuleList(EvoBlock(dn, de, dm, dtri, heads, tri_attn, opm_c) for _ in range(msa_layers))
        self.blocks = nn.ModuleList(Block(dn, de, dtri, heads, tri_attn) for _ in range(pair_layers))
        self.edge_head = nn.Sequential(nn.LayerNorm(de), nn.Linear(de, de), nn.GELU(), nn.Linear(de, 1))
        self.pi_head = nn.Sequential(nn.LayerNorm(dn), nn.Linear(dn, dn), nn.GELU(), nn.Linear(dn, 1))
        self.conf_head = nn.Sequential(nn.LayerNorm(dn), nn.Linear(dn, dn), nn.GELU(), nn.Linear(dn, 1))
        self.grad_ckpt = grad_ckpt

    def _run(self, fn, *args):
        if self.grad_ckpt and self.training:
            return checkpoint(fn, *args, use_reentrant=False)
        return fn(*args)

    def forward(self, ctx, b, n_recycle=None):
        NB = b["NB"]
        h = self.node_in(ctx["node_feat"])
        z = self.edge_in(ctx["edge_feat"])
        if self.use_ensemble:
            msa = self.msa_in(b["msa_f"]) + self.pos_emb(b["msa_ps"]) + self.pos_emb(b["msa_pp"])
            for blk in self.evo:
                msa, h, z = self._run(blk, msa, h, z, NB, ctx["rev_pos"], ctx["mutual"], ctx["tri"],
                                      b["ens_succ"], b["ens_pred"])
        for blk in self.blocks:
            h, z = self._run(blk, h, z, NB, ctx["rev_pos"], ctx["mutual"], ctx["tri"])
        logit = self.edge_head(z).squeeze(-1).float()
        mutual, rev_pos = ctx["mutual"], ctx["rev_pos"]
        sym = torch.where(mutual, 0.5 * (logit + logit[NB, rev_pos]), logit)
        return {"edge_logit": sym, "pi_f": self.pi_head(h).squeeze(-1).float(),
                "conf_logit": self.conf_head(h).squeeze(-1).float()}


def tourfold_losses(out, b, w_bce=0.5, w_pi=1.0, w_conf=0.2):
    loss, logs = compute_losses(out, b, w_bce, w_pi)
    if "conf_logit" in out and w_conf > 0:
        top2 = out["edge_logit"].detach().topk(2, dim=1).indices
        target = torch.gather(b["y"], 1, top2).all(1).float()
        lc = F.binary_cross_entropy_with_logits(out["conf_logit"], target)
        loss = loss + w_conf * lc
        logs["conf_bce"] = lc.item()
        logs["loss"] = loss.item()
    return loss, logs


def load_model(path, device):
    """Rebuild a TourFoldNet from a training checkpoint (its stored config) and load the weights."""
    ck = torch.load(path, map_location=device, weights_only=False)
    cfg = dict(ck["cfg"])
    cfg["grad_ckpt"] = False
    model = TourFoldNet(**cfg).to(device)
    model.load_state_dict(ck["model"])
    return model.eval(), ck


def rank_neighbours(NB, logit):
    """All graph neighbours of every node, best first by logit; equal logits (bf16 output) keep the graph's
    distance order, so the ranking is deterministic."""
    return torch.gather(NB, 1, torch.sort(logit, dim=1, descending=True, stable=True).indices)


@torch.no_grad()
def predict_tourfold(model, xy, K, device, ens_succ=None, ens_cost=None, k_out=5):
    """Candidates, integer pi, per-edge probabilities and per-node confidence for one instance.
    'rank' lists all K graph neighbours of each node, best first (see rank_neighbours)."""
    t0 = time.perf_counter()
    use = model.cfg.get("use_ensemble", False)
    prep = prepare_instance(xy, K, ens_succ=ens_succ if use else None, ens_cost=ens_cost if use else None,
                            extra=model.cfg.get("graph_extra", 0) if use else 0)
    b = to_device(collate([prep]), device)
    t1 = time.perf_counter()
    ctx = graph_context(b, k_tri=model.cfg.get("k_tri"))
    dt = autocast_dtype(device)
    model.eval()
    with torch.autocast(device_type=device.type, dtype=dt, enabled=dt is not None):
        out = model(ctx, b)
    logit = out["edge_logit"].float()
    top = logit.topk(k_out, dim=1).indices
    cand = torch.gather(b["NB"], 1, top).cpu().numpy()
    rank = rank_neighbours(b["NB"], logit).cpu().numpy()
    pi_hat = (b["s"] * out["pi_f"]).double()
    pi_hat = pi_hat - pi_hat.mean()
    pi_int = torch.round(pi_hat * PRECISION * prep["span"]).long().cpu().numpy()
    if device.type == "cuda":
        torch.cuda.synchronize()
    t2 = time.perf_counter()
    alpha = np.tile(np.arange(k_out, dtype=np.int64) * 100, (len(cand), 1))
    return {"cand": cand, "rank": rank, "alpha": alpha, "pi": pi_int, "prob": torch.sigmoid(logit).cpu().numpy(),
            "conf": torch.sigmoid(out["conf_logit"]).cpu().numpy(), "NB": prep["NB"], "D": prep["D"],
            "t_prep": t1 - t0, "t_model": t2 - t1, "t_total": t2 - t0}
'''}
for _name, _src in MODULES.items():
    with open(os.path.join(LOCAL, _name), "w") as _f:
        _f.write(_src)
if LOCAL not in sys.path:
    sys.path.insert(0, LOCAL)
import tsplab as T, tsp_model as M
importlib.reload(T); importlib.reload(M)
print(f"Modüller hazır: {LOCAL}")

## 2 · Kurulum ve ayarlar
Drive bağlantısı, klasörler, ortam bilgisi (vCPU, GPU), ayarlar ve kayıt yardımcıları. Örnek başına paralel koşu sayısı fiziksel çekirdek sayısıdır (12 vCPU'da 6). Yaklaşık 1 dakika.

In [ ]:
import re, json, time, math, glob, shutil, traceback, datetime, warnings
import numpy as np, pandas as pd, torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if TEST_MODE:
    ROOT = os.environ["TOURFOLD_ROOT"]
else:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/TourFold_TSP"
R2, R3, R4 = f"{ROOT}/v2", f"{ROOT}/v3", f"{ROOT}/v4"
RES2, RES3, RES4 = f"{R2}/results", f"{R3}/results", f"{R4}/results"
INST_DIR = f"{RES4}/instances"            # one file per finished instance: the unit of resuming
WORK = f"{LOCAL}/work"
for _d in (R4, RES4, INST_DIR, f"{R4}/figures", WORK, f"{LOCAL}/tsplib"):
    os.makedirs(_d, exist_ok=True)
LOG_PATH, STATUS_PATH, SUMMARY_PATH = f"{R4}/log_v4.txt", f"{R4}/status_v4.json", f"{RES4}/summary_v4.json"
T0 = time.time()


def log(msg):
    line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {msg}"
    print(line, flush=True)
    try:                       # a brief Google Drive hiccup must not end a many-hour run
        with open(LOG_PATH, "a") as f:
            f.write(line + "\n")
    except OSError:
        pass


def jdump(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1, ensure_ascii=False, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
    os.replace(tmp, path)


def csv_atomic(df, path):
    df.to_csv(path + ".tmp", index=False)
    os.replace(path + ".tmp", path)


def md(df, index=True):
    try:
        return df.to_markdown(index=index)
    except Exception:  # noqa: BLE001  (tabulate yoksa)
        return "```\n" + df.to_string(index=index) + "\n```"


def hfmt(sec):
    return f"{sec / 3600:.1f} saat" if sec >= 3600 else f"{max(1, round(sec / 60))} dk"


EST = {}   # step -> estimated remaining seconds


def status(step_name, state, **extra):
    try:
        jdump({"step": step_name, "state": state, "time": datetime.datetime.now().isoformat(timespec="seconds"),
               "elapsed_h": round((time.time() - T0) / 3600, 2),
               "remaining_h_estimate": round(sum(EST.values()) / 3600, 1), **extra}, STATUS_PATH)
    except OSError:
        pass


ENV = T.env_info()
NPROC = ENV["nproc"] or 1
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = M.autocast_dtype(DEV)
if DEV.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
STATE = {"stop": None}                    # set when a critical step fails; later experiment steps are skipped
SUMMARY = json.load(open(SUMMARY_PATH)) if os.path.exists(SUMMARY_PATH) else {}
SUMMARY.update(env=ENV, errors={})


def step(name, fn, *a, critical=True, **kw):
    """Run one step. Errors are logged and recorded instead of raised, so 'Run all' always reaches the report and the
    result package; after a critical error the remaining experiment steps are skipped."""
    if STATE["stop"]:
        log(f"» {name}: atlandı — {STATE['stop']}")
        return None
    t0 = time.time()
    log(f"» {name}")
    try:
        return fn(*a, **kw)
    except Exception as e:  # noqa: BLE001
        SUMMARY["errors"][name] = repr(e)
        log(f"HATA ({name}): {e!r}\n{traceback.format_exc()}")
        if critical:
            STATE["stop"] = f"'{name}' adımı hata verdi"
        return None
    finally:
        sm = SUMMARY.setdefault("step_minutes", {})
        sm[name] = round(sm.get(name, 0.0) + (time.time() - t0) / 60, 1)
        EST[name] = 0.0
        try:
            jdump(SUMMARY, SUMMARY_PATH)
        except OSError:
            pass


def load_json(path, what):
    if os.path.exists(path):
        return json.load(open(path))
    STATE["stop"] = f"{what} bulunamadı ({path}); v4 önceki çalıştırmaların Drive'daki verisini kullanır"
    return {}


CFG = load_json(f"{ROOT}/config.json", "ilk çalıştırmanın ayarları")
C2 = load_json(f"{R2}/config_v2.json", "v2 ayarları")
C4_PATH = f"{R4}/config_v4.json"
_w = max(1, NPROC // 2)                   # parallel LKH runs per instance: one per physical core
if TEST_MODE:
    _w = int(os.environ.get("TOURFOLD_TEST_WORKERS", _w))
C4 = {**dict(workers=_w, seed0=101, k_tf=3, k=5, pad=3.0, overhead_s=8.0),
      **(json.load(open(C4_PATH)) if os.path.exists(C4_PATH) else {})}
jdump(C4, C4_PATH)
W, KT, KC = int(C4["workers"]), int(C4["k_tf"]), int(C4["k"])
SEEDS = [int(C4["seed0"]) + i for i in range(W)]
log(f"TourFold v4 başlıyor | ortam {ENV} | cihaz {DEV} | örnek başına {W} paralel LKH koşusu (tohumlar {SEEDS}) | "
    f"veri {ROOT} | sonuçlar {R4}")
if 2 * W > NPROC and not TEST_MODE:
    log(f"UYARI: bu çalışma zamanında {NPROC} vCPU var ama örnek başına {W} paralel koşu ayarlı (önceki oturumdan). "
        "Koşular çekirdek paylaşacağından süreler bozulur; 12 vCPU'lu bir çalışma zamanı seçin.")
elif W < 4 and not TEST_MODE:
    log(f"UYARI: yalnız {NPROC} vCPU var; örnek başına {W} paralel koşu ve {W} turluk topluluk olacak (model 4–16 turla "
        "eğitildi). Mümkünse 12 vCPU'lu bir çalışma zamanı seçin.")
if DEV.type != "cuda" and not TEST_MODE:
    STATE["stop"] = ("GPU bulunamadı: TourFold'un tahmini bütçeden düşüldüğü için CPU'da haksız yavaş kalır. Çalışma "
                     "zamanını GPU'lu seçip (Çalışma zamanı ▸ Çalışma zamanı türünü değiştir) Tümünü çalıştır'ı yeniden "
                     "seçin")
if STATE["stop"]:
    log(f"DURDU: {STATE['stop']}")

## 3 · LKH-3 ve öz-sınama
LKH-3'ü önceki çalıştırmalardaki sürümüyle (3.0.14) derler ve bu defterde kullanılan üç LKH çağrısını küçük bir örnekte sınar: tam POPMUSIC koşusu, ilk deneme, ilk denemenin turundan yeni adaylarla devam. Yaklaşık 1–2 dakika.

In [ ]:
POP = {"CANDIDATE_SET_TYPE": "POPMUSIC", "SUBGRADIENT": "NO"}
SAME_FIRST = "ilk deneme, aynı tohumlu tam koşunun ilk denemesiyle aynı"


def lkh_build():
    if os.environ.get("TOURFOLD_LKH"):
        return os.environ["TOURFOLD_LKH"]
    ver = CFG.get("lkh_version")          # the version of the earlier runs
    have = glob.glob(f"{LOCAL}/lkh/LKH-{ver}/LKH") if ver else []
    lkh = have[0] if have else T.build_lkh(f"{LOCAL}/lkh", version=ver)
    log(f"LKH hazır: {lkh} (sürüm {ver or '?'})")
    return lkh


def selftest(lkh):
    """The three LKH calls of this notebook on a small clustered instance."""
    n = 60 if TEST_MODE else 400
    xy = T.to_int_coords(T.gen_instance(n, "clustered", np.random.default_rng(2026)))
    a = T.task_anytime(lkh, WORK, budget=3.0, seed=SEEDS[0], xy=xy, params={**POP, "MAX_TRIALS": 20}, timeout=60)
    f = T.task_first_trial(lkh, WORK, seed=SEEDS[0], xy=xy, timeout=60)
    nb, _ = T.knn(xy, 16)
    lst = T.mix_candidates(nb, f["cand"], KT, KC, fill=nb[:, KT:])
    g = T.task_guided(lkh, WORK, budget=2.0, seed=SEEDS[0], xy=xy, cand=lst, init_tour=f["tour"], timeout=60)
    checks = {"LKH-3 POPMUSIC koşusu iyileşme izi verdi": bool(a["events"]) and a["cost_min"] is not None,
              "ilk deneme geçerli bir tur ve aday listesi verdi": bool(T.is_valid_tour(f["tour"], n))
              and f["cand"].shape == (n, KC),
              SAME_FIRST: bool(a["events"]) and bool(f["events"]) and a["events"][0][1] == f["events"][0][1],
              "koşu, ilk denemenin turundan yeni adaylarla devam etti": g["cost_check"] is not None
              and g["cost_check"] <= f["cost"] + 0.5}
    for k, v in checks.items():
        log(("OK    " if v else ("BİLGİ " if k == SAME_FIRST else "HATA  ")) + k)
    if not checks[SAME_FIRST]:
        log("BİLGİ: ilk denemeler birebir aynı değil; karşılaştırma yine adil, yalnız koşu çiftleri aynı turdan başlamaz")
    SUMMARY["selftest"] = checks
    jdump(checks, f"{RES4}/selftest.json")
    bad = [k for k, v in checks.items() if not v and k != SAME_FIRST]
    if bad:
        raise RuntimeError("Öz-sınama başarısız: " + "; ".join(bad))


LKH = step("1-LKH", lkh_build)
step("1-öz-sınama", selftest, LKH)

## 4 · Örnekler ve TourFold-v2 modeli
v2–v3'teki 51 örneği (1.000, 10.000 ve 20.000 düğüm, TSPLIB) ve aynı süre bütçelerini hazırlar; TourFold-v2'yi Drive'dan yükler ve 6 turluk toplulukla aday kalitesini doğrulama örneklerinde kısaca ölçer. Yaklaşık 3–4 dakika.

In [ ]:
def eval_instances():
    """The evaluation instances of the earlier runs: synthetic ones regenerated from their seeds, TSPLIB from Drive."""
    def synth(prefix, n, count, dist, seed):
        rng = np.random.default_rng(seed)
        return [dict(name=f"{prefix}_{i:03d}", xy=T.to_int_coords(T.gen_instance(n, dist, rng)), ew="EUC_2D",
                     problem_file=None, opt=np.nan) for i in range(count)]
    sets = {"uni1k": synth("uni1k", CFG["size_small"], CFG["n_small"], "uniform", 101),
            "clu1k": synth("clu1k", CFG["size_small"], CFG["n_small"], "clustered", 202),
            "uni10k": synth("uni10k", CFG["size_large"], CFG["n_large"], "uniform", 303),
            "clu10k": synth("clu10k", CFG["size_large"], CFG["n_large"], "clustered", 404)}
    tsp = []
    if TEST_MODE and os.environ.get("TOURFOLD_TEST_NO_TSPLIB") != "1":
        for i, n in enumerate([110, 130]):
            pth = f"{LOCAL}/tsplib/fake{i}.tsp"
            T.write_tsp(pth, np.random.default_rng(i).random((n, 2)) * 1000, name=f"fake{i}")
            tsp.append(dict(name=f"fake{i}", xy=T.read_tsplib(pth)["xy"], ew="EUC_2D", problem_file=pth, opt=np.nan))
    elif os.environ.get("TOURFOLD_TEST_NO_TSPLIB") != "1":
        try:
            paths = T.download_tsplib(f"{ROOT}/tsplib_raw")
        except Exception as e:  # noqa: BLE001
            log(f"TSPLIB dosyaları okunamadı ({e!r})")
            paths = {}
        log(f"TSPLIB: {len(paths)} örnek dosyası hazır" if paths else
            "TSPLIB dosyaları bulunamadı; TSPLIB örnekleri bu çalıştırmada atlanıyor")
        for nm, path in sorted(paths.items()):
            inst = T.read_tsplib(path)
            if inst["edge_weight_type"] not in ("EUC_2D", "CEIL_2D") or not (1000 <= inst["dimension"] <= CFG["tsplib_max_n"]):
                continue
            local = f"{LOCAL}/tsplib/{nm}.tsp"
            shutil.copy(path, local)
            tsp.append(dict(name=nm, xy=inst["xy"], ew=inst["edge_weight_type"], problem_file=local,
                            opt=float(T.TSPLIB_OPT[nm])))
    sets["tsplib"] = sorted(tsp, key=lambda d: len(d["xy"]))
    return {k: v for k, v in sets.items() if v}


def load_ref(sets):
    """Reference tours of the evaluation instances (first run; read only)."""
    REF = {}
    for name, insts in sets.items():
        files = sorted(p for p in glob.glob(f"{ROOT}/eval/{name}_part*.npz") if not p.endswith(".tmp.npz"))
        order, recs, seen = {d["name"]: i for i, d in enumerate(insts)}, [], set()
        for fpath in files:
            sh = T.Shard(fpath)
            for i in range(len(sh)):
                r = sh[i]
                if r["name"] in order and r["name"] not in seen:
                    seen.add(r["name"])
                    recs.append(r)
        if recs:
            REF[name] = sorted(recs, key=lambda r: order[r["name"]])
    if not REF:
        raise RuntimeError(f"değerlendirme verisi bulunamadı ({ROOT}/eval)")
    return REF


def lkh_eval_selection(REF, sets):
    """The 51 instances and time budgets of the v2/v3 comparisons (same selection rule, same budgets)."""
    sel = []
    for name, key, bkey in (("uni1k", "any_1k", "b1k"), ("clu1k", "any_1k", "b1k"),
                            ("uni10k", "any_10k", "b10k"), ("clu10k", "any_10k", "b10k")):
        by = {d["name"]: d for d in sets.get(name, [])}
        for r in REF.get(name, [])[: C2[key]]:
            d = by[r["name"]]
            sel.append((name, dict(name=r["name"], xy=d["xy"], ew=d["ew"], problem_file=d["problem_file"],
                                   opt=float(r["opt"]), ref_cost=float(r["cost"])), float(C2[bkey])))
    by = {d["name"]: d for d in sets.get("tsplib", [])}
    for r in REF.get("tsplib", []):
        if r["n"] >= C2["tsplib_min_n"]:
            d = by[r["name"]]
            sel.append(("tsplib", dict(name=r["name"], xy=d["xy"], ew=d["ew"], problem_file=d["problem_file"],
                                       opt=float(r["opt"]), ref_cost=float(r["cost"])),
                        float(max(C2["tsplib_min"], C2["tsplib_per_node"] * r["n"]))))
    for dist, seed, pre in (("uniform", 505, "uni"), ("clustered", 606, "clu")):
        rng = np.random.default_rng(seed)
        for i in range(int(C2["n20k"])):
            xy = T.to_int_coords(T.gen_instance(int(C2["size20k"]), dist, rng))
            sel.append((f"{pre}20k", dict(name=f"{pre}20k_{i:03d}", xy=xy, ew="EUC_2D", problem_file=None,
                                          opt=float("nan"), ref_cost=float("inf")), float(C2["b20k"])))
    return sorted(sel, key=lambda x: x[2])


def load_instances():
    sets = eval_instances()
    sel = lkh_eval_selection(load_ref(sets), sets)
    counts = {}
    for name, _, _ in sel:
        counts[name] = counts.get(name, 0) + 1
    SUMMARY["instances"] = counts
    log(f"{len(sel)} örnek: {counts} | toplam bütçe {sum(B for *_, B in sel) / 3600:.2f} saat (yapılandırma başına)")
    return sel


def load_tf2():
    path = f"{R2}/runs/TourFold-v2/ckpt.pt"
    if not os.path.exists(path):
        raise RuntimeError(f"TourFold-v2 modeli bulunamadı ({path}); önce v2 defterini çalıştırın")
    model, ck = M.load_model(path, DEV)
    done = int(ck.get("epoch", -1)) + 1
    if done < int(C2.get("epochs", 12)):
        raise RuntimeError(f"TourFold-v2'nin eğitimi yarım ({done}/{C2.get('epochs')} epoch); önce v2 defterini tamamlayın")
    SUMMARY["training_v2_last"] = (ck.get("hist") or [{}])[-1]
    log(f"TourFold-v2 yüklendi: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M parametre, {done} epoch")
    return model


def rank_many(model, recs, m, bs=8, big=3000):
    """TourFold-v2 ranking (n, K) of every node's graph neighbours, best first, from the first m stored tours."""
    ex = model.cfg.get("graph_extra", 0)
    outs = [None] * len(recs)
    small = [i for i, r in enumerate(recs) if r["n"] <= big]
    model.eval()
    for s in range(0, len(small), bs):
        ids = small[s:s + bs]
        preps = [M.prepare_instance(recs[i]["xy"], CFG["K"], ens_succ=recs[i]["ens_succ"][:, :m],
                                    ens_cost=np.asarray(recs[i]["ens_cost"])[:m], extra=ex) for i in ids]
        b = M.to_device(M.collate(preps), DEV)
        ctx = M.graph_context(b, k_tri=model.cfg.get("k_tri"))
        with torch.no_grad(), torch.autocast(device_type=DEV.type, dtype=AMP, enabled=AMP is not None):
            out = model(ctx, b)
        rk = M.rank_neighbours(b["NB"], out["edge_logit"].float()).cpu().numpy()
        off = b["offsets"].cpu().numpy()
        for j, i in enumerate(ids):
            outs[i] = rk[off[j]:off[j + 1]] - off[j]
    for i, r in enumerate(recs):
        if outs[i] is None:
            outs[i] = M.predict_tourfold(model, r["xy"], CFG["K"], DEV, ens_succ=r["ens_succ"][:, :m],
                                         ens_cost=np.asarray(r["ens_cost"])[:m])["rank"]
    return outs


def model_check(tf2):
    """Candidate quality with W tours (the v4 ensemble size) next to 4 and 8 tours, on validation instances."""
    shards = sorted(p for p in glob.glob(f"{ROOT}/train/shard_*.npz") if not p.endswith(".tmp.npz"))
    if not shards:
        log("doğrulama örnekleri bulunamadı; model denetimi atlandı")
        return None
    sh = T.Shard(shards[-1])
    val = [sh[i] for i in range(min(len(sh), 10 if TEST_MODE else 300))]
    m_max = val[0]["ens_succ"].shape[1]
    out = {}
    for m in sorted({min(4, m_max), min(W, m_max), min(8, m_max)}):
        rk = rank_many(tf2, val, m)
        out[m] = float(np.mean([1000 * T.candidate_coverage(r_[:, :KC], v["tour"], ks=[KC])[KC]["missing"] / v["n"]
                                for r_, v in zip(rk, val)]))
    log("TourFold-v2, doğrulama: eksik tur kenarı / 1000 düğüm (ilk 5 aday): "
        + ", ".join(f"{m} tur {v:.2f}" for m, v in out.items()))
    SUMMARY["model_check"] = {str(k): v for k, v in out.items()}
    return out


SEL = step("2-örnekler", load_instances)
TF2 = step("2-model", load_tf2)
if TF2 is not None:
    step("2-model denetimi", model_check, TF2, critical=False)
if SEL and not STATE["stop"]:
    _todo = [x for x in SEL if not os.path.exists(f"{INST_DIR}/{x[1]['name']}.json")]
    EST["3-çok çekirdekli LKH"] = sum(2 * B + C4["overhead_s"] for *_, B in _todo)
    log(f"Kaba tahmin: kalan ~{hfmt(sum(EST.values()))} ({len(_todo)}/{len(SEL)} örnek kaldı; son tahmin log_v4.txt "
        "ve status_v4.json içinde)")

## 5 · Çok çekirdekli LKH karşılaştırması (asıl deney)
Her örnek için önce taban, sonra TourFold koşuları (sıra örnekten örneğe değişir). Biten her örnek Drive'a yazılır; kesilirse kaldığı örnekten devam eder. Log her örnekte iki tarafın en iyi turunu ve kalan süreyi yazar. Yaklaşık 3,7 saat.

In [ ]:
import concurrent.futures as cf, concurrent.futures.process
import multiprocessing as mp


class LKHPool:
    """W persistent worker processes, one LKH run each at a time. Starting processes stays out of the timings."""

    def __init__(self, n):
        self.n, self.ex = n, None

    def start(self):
        if self.ex is None:
            self.ex = cf.ProcessPoolExecutor(max_workers=self.n, mp_context=mp.get_context("spawn"))
            pids = [f.result() for f in [self.ex.submit(T.worker_ping, 1.0) for _ in range(self.n)]]
            log(f"{len(set(pids))} işçi süreci hazır")

    def run(self, jobs):
        """jobs: list of (function, kwargs); results in order, errors come back as {'error': ...}."""
        for _ in range(2):
            try:
                self.start()
                futs = [self.ex.submit(T._safe_call, (fn, kw)) for fn, kw in jobs]
                return [f.result() for f in futs]
            except cf.process.BrokenProcessPool:
                log("UYARI: işçi süreçleri çöktü; yeniden başlatılıyor")
                self.close()
        raise RuntimeError("işçi süreçleri iki kez çöktü")

    def close(self):
        if self.ex is not None:
            self.ex.shutdown(wait=False, cancel_futures=True)
            self.ex = None


def events_from(r, t0):
    """(cost, seconds since t0) for every improvement of an LKH run (its own clock starts at process launch)."""
    return T.abs_events(r, offset=r["t_start"] - t0)


def run_pop(pool, lkh, d, B):
    """Baseline: W independent LKH-3 POPMUSIC runs in parallel (one per core), each for the whole budget."""
    common = dict(lkh_bin=lkh, workroot=WORK, xy=d["xy"], problem_file=d["problem_file"], ew=d["ew"])
    t0 = time.time()
    res = pool.run([(T.task_anytime, dict(common, budget=B, seed=s, params=POP, timeout=B + C4["pad"])) for s in SEEDS])
    runs = []
    for s, r in zip(SEEDS, res):
        if "error" in r:
            log(f"LKH hatası {d['name']} / POPMUSIC / tohum {s}: {r['error']}")
            continue
        runs.append(dict(seed=s, events=events_from(r, t0), first=(r["events"][0][1] if r["events"] else None),
                         pre_time=r["pre_time"], start_delay=r["t_start"] - t0, timed_out=r["timed_out"]))
    return {"runs": runs, "wall": time.time() - t0}


def run_tourfold(pool, lkh, tf2, d, B):
    """TourFold: the same W runs stop after their first trial; TourFold-v2 reads the W tours as an ensemble; each run
    resumes from its own tour with TourFold's first KT candidates followed by its own POPMUSIC list. Everything
    between the start and the resumed runs (waiting for the slowest first trial, the model, the lists) counts."""
    common = dict(lkh_bin=lkh, workroot=WORK, xy=d["xy"], problem_file=d["problem_file"], ew=d["ew"])
    t0 = time.time()
    ph1 = pool.run([(T.task_first_trial, dict(common, seed=s, timeout=B + C4["pad"])) for s in SEEDS])
    t_ph1 = time.time() - t0
    ok = []
    for s, r in zip(SEEDS, ph1):
        if "error" in r:
            log(f"LKH hatası {d['name']} / ilk deneme / tohum {s}: {r['error']}")
        else:
            ok.append((s, r))
    if len(ok) < 2:
        raise RuntimeError(f"{d['name']}: ilk denemelerin çoğu başarısız")
    succ = np.stack([T.tour_succ_pred(r["tour"])[0] for _, r in ok], 1)
    costs = np.array([r["cost"] for _, r in ok], dtype=float)
    pr = M.predict_tourfold(tf2, d["xy"], CFG["K"], DEV, ens_succ=succ, ens_cost=costs)
    t1 = time.time()
    lists = [T.mix_candidates(pr["rank"], r["cand"], KT, KC, fill=pr["rank"][:, KT:]) for _, r in ok]
    t_list = time.time() - t1
    off = time.time() - t0
    bl = max(0.5, B - off)
    res = pool.run([(T.task_guided, dict(common, budget=bl, seed=s, cand=lst, init_tour=r["tour"], timeout=bl + C4["pad"]))
                    for (s, r), lst in zip(ok, lists)])
    runs, delays = [], []
    for (s, r1), r2 in zip(ok, res):
        ev = events_from(r1, t0) or [(r1["cost"], r1["t_start"] - t0 + r1["wall"])]   # trace missing: tour at exit
        if "error" in r2:
            log(f"LKH hatası {d['name']} / TourFold / tohum {s}: {r2['error']}")
        else:
            ev = ev + events_from(r2, t0)
            delays.append(r2["t_start"] - t0)
        runs.append(dict(seed=s, events=ev, first=(r1["events"][0][1] if r1["events"] else None),
                         pre_time=r1["pre_time"], first_wall=r1["wall"], resumed="error" not in r2,
                         timed_out=r2.get("timed_out")))
    in_pop = float(np.mean([(lst[:, :KT, None] == r["cand"][:, None, :]).any(-1).mean() for (_, r), lst in zip(ok, lists)]))
    meta = dict(t_phase1=t_ph1, t_inf=pr["t_total"], t_list=t_list, offset=off,
                resume_at=float(np.mean(delays)) if delays else float("nan"), m=len(ok), first_costs=costs.tolist(),
                tf_top_in_pop=in_pop)
    return {"runs": runs, "meta": meta, "wall": time.time() - t0}


def lkh_multicore(lkh, tf2, sel):
    todo = [(k, x) for k, x in enumerate(sel) if not os.path.exists(f"{INST_DIR}/{x[1]['name']}.json")]
    w_all = sum(2 * B + C4["overhead_s"] for _, (_, _, B) in todo)
    EST["3-çok çekirdekli LKH"] = w_all
    log(f"Çok çekirdekli karşılaştırma: {len(sel)} örnek ({len(todo)} kaldı), örnek başına 2 × {W} LKH koşusu | "
        f"bu adım için tahmini ~{hfmt(w_all)}")
    if not todo:
        return 0
    x0 = todo[0][1][1]["xy"]          # warm-up: CUDA start-up must not enter the timings
    M.predict_tourfold(tf2, x0, CFG["K"], DEV, ens_succ=np.stack([np.roll(np.arange(len(x0)), -1)] * W, 1),
                       ens_cost=np.ones(W))
    pool = LKHPool(W)
    t_stage, w_done = time.time(), 0.0
    try:
        pool.start()
        for j, (k, (name, d, B)) in enumerate(todo):
            status("3-çok çekirdekli LKH", "running", instance=d["name"], done=j, of=len(todo))
            order = ["pop", "tf"] if k % 2 == 0 else ["tf", "pop"]   # alternate the order against slow drifts
            out = dict(set=name, inst=d["name"], n=int(len(d["xy"])), budget=B, opt=float(d["opt"]),
                       ref_cost=float(d["ref_cost"]), order=order, workers=W, seeds=SEEDS)
            try:
                for conf in order:
                    out[conf] = run_pop(pool, lkh, d, B) if conf == "pop" else run_tourfold(pool, lkh, tf2, d, B)
            except Exception as e:  # noqa: BLE001  (one bad instance must not end the experiment)
                SUMMARY["errors"][f"örnek {d['name']}"] = repr(e)
                log(f"HATA ({d['name']}): {e!r}\n{traceback.format_exc()}")
                continue
            jdump(out, f"{INST_DIR}/{d['name']}.json")
            w_done += 2 * B + C4["overhead_s"]
            EST["3-çok çekirdekli LKH"] = (time.time() - t_stage) / w_done * max(0.0, w_all - w_done)
            best_c = {c: min([e[0] for r in out[c]["runs"] for e in r["events"] if e[1] <= B] or [np.inf])
                      for c in ("pop", "tf")}
            rel = 100 * (best_c["tf"] - best_c["pop"]) / best_c["pop"] if np.isfinite(best_c["pop"]) else float("nan")
            log(f"{j + 1}/{len(todo)} {d['name']} (n={out['n']}, bütçe {B:.0f} sn): bütçe sonunda en iyi tur "
                f"POPMUSIC×{W} {best_c['pop']:.0f}, TourFold×{W} {best_c['tf']:.0f} ({rel:+.4f}%) | TourFold'un devam "
                f"anı {out['tf']['meta']['offset']:.1f} sn | bu adım için kalan ~{hfmt(EST['3-çok çekirdekli LKH'])}")
    finally:
        pool.close()
    return len(todo)


if SEL and TF2 is not None:
    step("3-çok çekirdekli LKH", lkh_multicore, LKH, TF2, SEL)
else:
    log("» 3-çok çekirdekli LKH: atlandı — " + (STATE["stop"] or "örnekler ya da model yüklenemedi"))

## 6 · Analiz ve rapor
Tabloları, grafikleri ve `RAPOR_v4.md`'yi üretir: paralel koşuların en iyisiyle süre–kalite, kazanç/beraberlik/kayıp ve işaret testi, aynı tohumlu koşuların karşılaştırması, TourFold'un ek süresi. Deney yarım kalsa da biten örneklerle çalışır.

In [ ]:
FRACS, TARGETS = (0.1, 0.25, 0.5, 1.0), (1.0, 0.5, 0.2, 0.1, 0.05)
CONFS = {"pop": f"LKH-3 POPMUSIC ×{W}", "tf": f"TourFold ×{W}"}
CLASSES = ["1k", "10k", "TSPLIB", "20k"]
RESULTS = {}


def size_class(s):
    return "TSPLIB" if s == "tsplib" else s[3:] if s[:3] in ("uni", "clu") else s


def earlier_runs():
    """Single-core v2 and v3 runs on the same instances (for the best-known costs and the context table)."""
    out = {}
    for tag, pat in (("v2", f"{RES2}/lkh_eval_parts/part_*.json"), ("v3", f"{RES3}/lkh_eval_parts/part_*.json")):
        out[tag] = []
        for f in sorted(glob.glob(pat)):
            try:
                out[tag] += json.load(open(f))
            except Exception:  # noqa: BLE001
                pass
    return out


def sign_p(w, l_):
    """Two-sided sign test, ties dropped."""
    if w + l_ == 0:
        return float("nan")
    from scipy.stats import binomtest
    return float(binomtest(int(w), int(w + l_), 0.5).pvalue)


def wilcoxon_p(diff):
    from scipy.stats import wilcoxon
    d = np.asarray([x for x in diff if np.isfinite(x) and abs(x) > 1e-12])
    return float(wilcoxon(d).pvalue) if len(d) >= 6 else float("nan")


def compare(a, b):
    """Win/tie/loss of a against b (smaller gap wins; having a tour beats having none)."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    w = int(((a < b - 1e-12) | (~np.isnan(a) & np.isnan(b))).sum())
    l_ = int(((a > b + 1e-12) | (np.isnan(a) & ~np.isnan(b))).sum())
    return w, len(a) - w - l_, l_


def fmt_p(v):
    return "nan" if v != v else f"{v:.3g}"


def time_to_cost(events, c):
    """First moment the best tour so far costs at most c (inf if never)."""
    for cost, t in sorted(events, key=lambda e: e[1]):
        if cost <= c + 1e-9:
            return float(t)
    return float("inf")


def analyse():
    files = sorted(glob.glob(f"{INST_DIR}/*.json"))
    if not files:
        raise RuntimeError("henüz biten örnek yok")
    inst = sorted((json.load(open(f)) for f in files), key=lambda x: (x["budget"], x["inst"]))
    prev = earlier_runs()
    best = {}
    for x in inst:                     # optimum for TSPLIB; else the best of the reference tour and all runs
        if np.isfinite(x["opt"]):
            best[x["inst"]] = float(x["opt"])
            continue
        c = [x["ref_cost"]] + [e[0] for k in CONFS for r in x[k]["runs"] for e in r["events"]]
        c += [e[0] for tag in prev for r in prev[tag] if r.get("inst") == x["inst"] and int(r["n"]) == x["n"]
              for e in r["events"]]
        best[x["inst"]] = float(min(c))
    rows, prow, orow, reach = [], [], [], []
    for x in inst:
        b, B = best[x["inst"]], x["budget"]
        for k, cname in CONFS.items():
            ev = [e for r in x[k]["runs"] for e in r["events"]]          # the best of the W runs at every moment
            g = T.curve_abs(ev, b, [f * B for f in FRACS])
            rows.append(dict(set=x["set"], cls=size_class(x["set"]), inst=x["inst"], n=x["n"], config=cname, budget=B,
                             **{f"gap@{int(f * 100)}%B": v for f, v in zip(FRACS, g)},
                             **{f"time_to_{t}%": T.time_to_gap_abs(ev, b, t) for t in TARGETS}))
            for r in x[k]["runs"]:
                g2 = T.curve_abs(r["events"], b, [0.5 * B, B])
                prow.append(dict(set=x["set"], cls=size_class(x["set"]), inst=x["inst"], seed=r["seed"], config=k,
                                 **{"gap@50%B": g2[0], "gap@100%B": g2[1]}, first=r.get("first")))
        evs = {k: [e for r in x[k]["runs"] for e in r["events"]] for k in CONFS}
        fin = {k: min([c for c, t in evs[k] if t <= B] or [np.inf]) for k in CONFS}
        reach.append(dict(set=x["set"], cls=size_class(x["set"]), inst=x["inst"],
                          tf_to_pop=time_to_cost(evs["tf"], fin["pop"]) / B if np.isfinite(fin["pop"]) else np.nan,
                          pop_to_tf=time_to_cost(evs["pop"], fin["tf"]) / B if np.isfinite(fin["tf"]) else np.nan))
        m = x["tf"]["meta"]
        resume = m.get("resume_at", m["offset"])
        orow.append(dict(set=x["set"], inst=x["inst"], budget=B, ilk_denemeler=m["t_phase1"], model=m["t_inf"],
                         liste=m["t_list"], devam_anı=resume, bütçe_payı_yüzde=100 * resume / B,
                         taban_başlama_gecikmesi=float(np.mean([r.get("start_delay", 0.0) for r in x["pop"]["runs"]]
                                                              or [np.nan])),
                         tf_ilk3_popmusicte=m.get("tf_top_in_pop")))
    df, dr, do = pd.DataFrame(rows), pd.DataFrame(prow), pd.DataFrame(orow)
    csv_atomic(df, f"{RES4}/lkh_v4_best_of_runs.csv")
    csv_atomic(dr, f"{RES4}/lkh_v4_per_run.csv")
    csv_atomic(do, f"{RES4}/lkh_v4_tourfold_overhead.csv")
    gcols = [f"gap@{int(f * 100)}%B" for f in FRACS]
    gap_tab = df.groupby(["set", "config"], sort=False)[gcols].mean()
    cnt = df.groupby(["set", "config"], sort=False)[["gap@10%B", "gap@25%B"]].count()
    cnt.columns = ["turlu@10%B", "turlu@25%B"]
    gap_tab = gap_tab.join(cnt)
    gap_tab["örnek"] = df.groupby(["set", "config"], sort=False).size()
    t_tab = df.groupby(["set", "config"], sort=False)[[f"time_to_{t}%" for t in TARGETS]].median()
    piv = {col: df.pivot_table(index="inst", columns="config", values=col, aggfunc="first", dropna=False)
           for col in gcols}
    cls_of = df.drop_duplicates("inst").set_index("inst")["cls"]
    pooled = []
    for col in ("gap@25%B", "gap@50%B", "gap@100%B"):
        p_ = piv[col]
        rec = {"bütçe": "%" + col[4:-2]}
        W_, T_, L_ = 0, 0, 0
        for c in CLASSES:
            ids = [i for i in p_.index if cls_of[i] == c]
            if ids:
                w, t, l_ = compare(p_.loc[ids, CONFS["tf"]], p_.loc[ids, CONFS["pop"]])
                rec[c] = f"{w}/{t}/{l_}"
                W_, T_, L_ = W_ + w, T_ + t, L_ + l_
        rec["tümü"] = f"{W_}/{T_}/{L_}"
        rec["işaret testi p"] = fmt_p(sign_p(W_, L_))
        rec["Wilcoxon p"] = fmt_p(wilcoxon_p(p_[CONFS["tf"]] - p_[CONFS["pop"]]))
        rec["ort. fark (puan)"] = round(float(np.nanmean(p_[CONFS["tf"]] - p_[CONFS["pop"]])), 4)
        pooled.append(rec)
    pooled = pd.DataFrame(pooled)
    csv_atomic(pooled, f"{RES4}/lkh_v4_wtl.csv")
    pr = dr.pivot_table(index=["inst", "seed"], columns="config", values=["gap@50%B", "gap@100%B", "first"],
                        aggfunc="first", dropna=False)
    paired = []
    for col in ("gap@50%B", "gap@100%B"):
        if ("tf" in pr[col]) and ("pop" in pr[col]):
            w, t, l_ = compare(pr[col]["tf"], pr[col]["pop"])
            paired.append({"bütçe": "%" + col[4:-2], "koşu çifti": len(pr), "K/B/K": f"{w}/{t}/{l_}",
                           "işaret testi p": fmt_p(sign_p(w, l_)),
                           "Wilcoxon p": fmt_p(wilcoxon_p(pr[col]["tf"] - pr[col]["pop"]))})
    paired = pd.DataFrame(paired)
    same_first = float("nan")
    if "first" in pr and "tf" in pr["first"] and "pop" in pr["first"]:
        a_, b_ = pr["first"]["tf"], pr["first"]["pop"]
        ok = a_.notna() & b_.notna()
        same_first = float((a_[ok] == b_[ok]).mean()) if ok.any() else float("nan")
    over = do.groupby("set", sort=False)[["ilk_denemeler", "model", "liste", "devam_anı", "bütçe_payı_yüzde",
                                          "taban_başlama_gecikmesi", "tf_ilk3_popmusicte"]].mean()
    ctx = []      # context: one single-core LKH-3 POPMUSIC run (v3, seed 101) against the best of W runs (v4)
    for r in prev.get("v3", []):
        if r.get("key") == "pop" and r["inst"] in best:
            ctx.append(dict(set=r["set"], inst=r["inst"], v3_tek=T.curve_abs(r["events"], best[r["inst"]], [r["budget"]])[0]))
    ctx = pd.DataFrame(ctx)
    if len(ctx):
        last = df[df["config"] == CONFS["pop"]].set_index("inst")["gap@100%B"]
        last_tf = df[df["config"] == CONFS["tf"]].set_index("inst")["gap@100%B"]
        ctx["v4_pop"] = ctx["inst"].map(last)
        ctx["v4_tf"] = ctx["inst"].map(last_tf)
        ctx = ctx.dropna(subset=["v4_pop"]).groupby("set", sort=False)[["v3_tek", "v4_pop", "v4_tf"]].mean()
        ctx.columns = ["tek koşu, POPMUSIC (v3)", f"{W} koşunun en iyisi, POPMUSIC", f"{W} koşunun en iyisi, TourFold"]
    rc = pd.DataFrame(reach)
    rtab = []
    for c in CLASSES + ["tümü"]:
        sub = rc if c == "tümü" else rc[rc["cls"] == c]
        if len(sub):
            rtab.append({"sınıf": c, "örnek": len(sub),
                         "TourFold → tabanın son kalitesi: medyan süre (bütçe %)": 100 * float(np.nanmedian(sub["tf_to_pop"].clip(upper=9.99))),
                         "TourFold bütçe içinde ulaştı (%)": 100 * float((sub["tf_to_pop"] <= 1.0 + 1e-9).mean()),
                         "taban → TourFold'un son kalitesi: medyan süre (bütçe %)": 100 * float(np.nanmedian(sub["pop_to_tf"].clip(upper=9.99))),
                         "taban bütçe içinde ulaştı (%)": 100 * float((sub["pop_to_tf"] <= 1.0 + 1e-9).mean())})
    rtab = pd.DataFrame(rtab)
    csv_atomic(rc, f"{RES4}/lkh_v4_time_to_rival.csv")
    out = {"n_done": len(inst), "gap": gap_tab.reset_index().to_dict("records"),
           "reach": rtab.to_dict("records"),
           "time_to_gap": t_tab.reset_index().to_dict("records"), "pooled": pooled.to_dict("records"),
           "paired": paired.to_dict("records"), "same_first": same_first,
           "overhead": over.reset_index().to_dict("records"),
           "context": ctx.reset_index().to_dict("records") if len(ctx) else [], "best": best}
    jdump(out, f"{RES4}/lkh_v4.json")
    log(f"v4 — {W} koşunun en iyisi, en iyi bilinene göre fark (%):\n" + gap_tab.round(4).to_string())
    log("v4 — TourFold vs POPMUSIC, kazanç/beraberlik/kayıp:\n" + pooled.to_string(index=False))
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            fr = np.linspace(0.02, 1.0, 60)
            for sname in dict.fromkeys(df["set"]):
                fig, ax = plt.subplots(figsize=(7.5, 4.2))
                for k, cname in CONFS.items():
                    xs = [x for x in inst if x["set"] == sname]
                    arr = np.array([T.curve_abs([e for r in x[k]["runs"] for e in r["events"]], best[x["inst"]],
                                                fr * x["budget"]) for x in xs], dtype=float)
                    ax.plot(100 * fr, np.where(np.isnan(arr).any(0), np.nan, arr.mean(0)), label=cname,
                            ls="--" if k == "pop" else "-")
                ax.set_yscale("symlog", linthresh=0.01)
                ax.set_xlabel("bütçenin yüzdesi (toplam süre)")
                ax.set_ylabel(f"{W} koşunun en iyisi: ortalama fark (%)")
                ax.set_title(f"v4 · {sname}: süre–kalite ({len(xs)} örnek)"); ax.legend(fontsize=8)
                plt.tight_layout(); fig.savefig(f"{R4}/figures/anytime_v4_{sname}.png", dpi=130); plt.close(fig)
    except Exception as e:  # noqa: BLE001
        log(f"grafik çizilemedi: {e!r}")
    return out


def write_report():
    L = ["# TourFold TSP v4 — çok çekirdekli karşılaştırma (otomatik rapor)", "",
         f"- Tarih: {datetime.datetime.now():%Y-%m-%d %H:%M}",
         f"- Ortam: {ENV}",
         f"- Örnek başına {W} paralel LKH koşusu (tohumlar {SEEDS}); TourFold listesi: modelin ilk {KT} adayı + koşunun "
         f"kendi POPMUSIC listesi ({KC} aday)",
         f"- Adım süreleri (dk, tüm oturumların toplamı): {SUMMARY.get('step_minutes', {})}", "",
         "## 1. Düzen", "",
         f"- **{CONFS['pop']}**: {W} bağımsız LKH-3 POPMUSIC koşusu (ascent yok), her biri bir çekirdekte, bütçenin "
         "tamamı boyunca.",
         f"- **{CONFS['tf']}**: aynı {W} koşu ilk denemeden sonra durur (aynı tohum, aynı ilk deneme). TourFold-v2 bu "
         f"{W} turu topluluk olarak okur. Her koşu kendi turundan, TourFold'un ilk {KT} adayı + kendi POPMUSIC "
         "listesiyle devam eder. En yavaş ilk denemeyi bekleme, model ve liste süresi bütçeden düşülür.",
         f"- Her an {W} koşunun en iyi turu sayılır. En iyi bilinen: TSPLIB'de optimum; diğerlerinde referans tur ile "
         "v2, v3 ve v4 koşularının en iyisi. Örnek bazında iki yapılandırmanın sırası değişir.", ""]
    st = SUMMARY.get("selftest")
    if st:
        bad = [k for k, v in st.items() if not v]
        L += ["## 2. Öz-sınama ve model denetimi", "",
              "Öz-sınama: tüm kontroller geçti." if not bad else "Öz-sınamada geçmeyen: " + ", ".join(bad)]
        mc = SUMMARY.get("model_check")
        if mc:
            L += ["", "TourFold-v2, doğrulama örneklerinde eksik tur kenarı / 1000 düğüm (ilk 5 aday): "
                  + ", ".join(f"{m} tur: {v:.2f}" for m, v in mc.items())]
        L.append("")
    r = RESULTS.get("lkh")
    if r:
        n_all = sum((SUMMARY.get("instances") or {}).values()) or r["n_done"]
        L += [f"## 3. Sonuçlar ({r['n_done']}/{n_all} örnek)", "",
              f"{W} koşunun en iyisinin, bütçenin %10, %25, %50 ve %100'ünde en iyi bilinene göre farkı (%, ortalama; "
              "nan = o ana kadar tur yok):", "", md(pd.DataFrame(r["gap"]).round(4), index=False), "",
              "Hedef farka ulaşma süresi (sn, medyan; inf = bütçe içinde ulaşılamadı):", "",
              md(pd.DataFrame(r["time_to_gap"]).round(2), index=False), "",
              f"### TourFold ×{W} ile LKH-3 POPMUSIC ×{W}", "",
              "Örnek bazında kazanç/beraberlik/kayıp (K/B/K = TourFold iyi / eşit / kötü), boyut sınıflarına göre ve "
              "tümü; p: iki yönlü işaret testi (beraberlikler hariç) ve Wilcoxon işaretli sıra testi; ortalama fark "
              "yüzde puan (eksi = TourFold iyi):", "", md(pd.DataFrame(r["pooled"]), index=False), ""]
        if r.get("reach"):
            L += ["Rakibin bütçe sonundaki kalitesine ulaşma süresi (bütçenin yüzdesi, medyan; 999 = bütçe içinde "
                  "ulaşamadı) ve bütçe içinde ulaşılan örneklerin oranı:", "",
                  md(pd.DataFrame(r["reach"]).round(1), index=False), ""]
        if r.get("paired"):
            L += [f"Koşu bazında (aynı örnek, aynı tohum, ilk deneme ortak; her koşu tek başına):", "",
                  md(pd.DataFrame(r["paired"]), index=False), ""]
        if r.get("same_first") == r.get("same_first"):
            L += [f"İlk denemesi iki yapılandırmada birebir aynı olan koşu oranı: {r['same_first']:.3f}", ""]
        L += ["### TourFold'un ek süresi (sn, ortalama; bütçeden düşülür)", "",
              md(pd.DataFrame(r["overhead"]).round(3), index=False), "",
              "ilk_denemeler: en yavaş ilk denemenin bitişi · model, liste: TourFold-v2'nin tahmini ve listelerin "
              "kurulması · devam_anı: koşuların TourFold adaylarıyla yeniden başladığı an · taban_başlama_gecikmesi: "
              "taban koşularının başlama gecikmesi · tf_ilk3_popmusicte: TourFold'un ilk 3 adayının koşunun kendi "
              "POPMUSIC listesinde de bulunma oranı.", ""]
        if r.get("context"):
            L += ["### Bağlam: tek koşu (v3) ve paralel koşular, bütçenin tamamında fark (%, ortalama)", "",
                  md(pd.DataFrame(r["context"]).round(4), index=False), ""]
    if SUMMARY.get("errors"):
        L += ["## Hatalar", ""] + [f"- {k}: {v}" for k, v in SUMMARY["errors"].items()] + [""]
    L += ["## Dosyalar", "",
          "- v4/results: tablolar (csv, json); instances/: örnek başına tüm koşuların izleri · v4/figures: grafikler · "
          "v4/log_v4.txt: ayrıntılı kayıt", ""]
    text = "\n".join(L)
    with open(f"{R4}/RAPOR_v4.md", "w") as fh:
        fh.write(text)
    jdump(SUMMARY, SUMMARY_PATH)
    return text


if glob.glob(f"{INST_DIR}/*.json"):
    _stop, STATE["stop"] = STATE["stop"], None       # the analysis also runs after an earlier error
    RESULTS["lkh"] = step("4-analiz", analyse, critical=False)
    STATE["stop"] = _stop
try:
    REPORT = write_report()
    log(f"Rapor yazıldı: {R4}/RAPOR_v4.md")
except Exception as _e:  # noqa: BLE001
    REPORT = None
    log(f"rapor yazılamadı: {_e!r}")

## 7 · Sonuç paketi
Sonuçları `TourFold_v4_sonuclar.zip` olarak Drive'a kaydeder ve tarayıcınıza indirir.

In [ ]:
def bundle_results(download=True):
    import zipfile
    tmp = f"{LOCAL}/TourFold_v4_sonuclar.zip"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for rel in ("RAPOR_v4.md", "log_v4.txt", "status_v4.json", "config_v4.json"):
            if os.path.exists(f"{R4}/{rel}"):
                z.write(f"{R4}/{rel}", rel)
        for pth, arc in ((f"{ROOT}/config.json", "config.json"), (f"{R2}/config_v2.json", "config_v2.json")):
            if os.path.exists(pth):
                z.write(pth, arc)
        for sub in ("results", "figures"):
            for p_ in sorted(glob.glob(f"{R4}/{sub}/**/*", recursive=True)):
                if os.path.isfile(p_) and not p_.endswith((".tmp", ".part")):
                    z.write(p_, os.path.relpath(p_, R4))
    shutil.copy(tmp, f"{R4}/TourFold_v4_sonuclar.zip")
    msg = f"Sonuç paketi Drive'da: {R4}/TourFold_v4_sonuclar.zip"
    if download and not TEST_MODE:
        import threading
        from google.colab import files
        th = threading.Thread(target=files.download, args=(tmp,), daemon=True)
        th.start()
        th.join(timeout=120)
        msg += " (tarayıcınıza da indiriliyor)"
    log(msg)


_n_done = len(glob.glob(f"{INST_DIR}/*.json"))
_n_all = sum((SUMMARY.get("instances") or {}).values())
if STATE["stop"] or SUMMARY.get("errors"):
    status("done_with_errors", "finished", errors=list(SUMMARY.get("errors", {})) or [STATE["stop"]],
           instances_done=_n_done, of=_n_all)
    log(f"Bitti, ancak hata var ({STATE['stop'] or 'ayrıntı: RAPOR_v4.md > Hatalar'}). {R4}/RAPOR_v4.md ve "
        "log_v4.txt dosyalarını (ya da zip'i) bana gönderin.")
elif _n_all and _n_done < _n_all:
    status("incomplete", "finished", instances_done=_n_done, of=_n_all)
    log(f"{_n_done}/{_n_all} örnek bitti. Tümünü çalıştır'ı yeniden seçerseniz kalanlar çözülür.")
else:
    status("done", "finished", instances_done=_n_done, of=_n_all)
    log(f"BİTTİ. Rapor: {R4}/RAPOR_v4.md — bu dosyayı (ya da TourFold_v4_sonuclar.zip'i) bana gönderin.")
try:
    bundle_results()
except Exception as _e:  # noqa: BLE001
    log(f"sonuç paketi hazırlanamadı: {_e!r}")
print("\n" + (REPORT or ""))